# Image Editing LLM Pipeline — Phase 2: SD Inpainting Bridge Training
**Spec version:** 1.7 · **Notebook version:** 1.7 · **Target runtime:** Google Colab A100

**Phase in this notebook:** Phase 2 — Stable Diffusion 1.5 inpainting bridge training
with a VLM projection adapter and UNet LoRA.

**Prerequisite artifacts (from Phase 0 + 1):**
- `data/imgedit_subset/samples_filtered.json` — filtered manifest with `shard_id` / `row_index` fields
- `data/imgedit_subset/vlm_hidden_states/shard_*.pt` — pre-cached VLM hidden states
- `data/imgedit_subset/vlm_hidden_states/hidden_states_meta.json` — template hash + shard count
- `checkpoints/phase1_vlm/` — Phase 1 LoRA adapter (not loaded in Phase 2; kept for reference)

**Training path (Phase 2):**
```
pre-cached VLM hidden states  ─► VLMProjectionAdapter ─► (batch, 1, 768)
                                                               │
CLIP(instruction)             ──────────────────────────► (batch, 77, 768)  ─cat─► (batch, 78, 768)
                                                                                        │
source image ─► VAE.encode ─► masked_src_latent (4ch)                                  │
target image ─► VAE.encode ─► noisy_tgt_latent  (4ch)  ─────────────────────────► UNet(9ch)
RLE mask     ─► decode     ─► mask_1ch                                                  │
                                                                                    noise_pred
                                                                                        │
                                                                                  MSE(noise_pred, noise)
```

**Inference path (Phase 3 — NOT implemented here):**
```
VLM(image + instruction) ─► hidden state ─► VLMProjectionAdapter
VLM bbox prediction      ─► SAM2          ─► mask
```

**Key constraints:**
- Phase 2 training uses GROUND-TRUTH RLE masks — NOT SAM2 (inference only)
- VLM hidden states are pre-cached — VLM is NOT loaded in Phase 2
- Prompt template must be byte-identical to Phase 1 (PHASE1_TEMPLATE_HASH)
- 9-channel UNet input: noisy_tgt(4) | mask(1) | masked_src(4)
- Combined conditioning: CLIP(77,768) concatenated with VLM_proj(1,768) = (78,768)


## §0.0 — v1.7 changelog (read first)

**What changed vs. v1.6**

- LoRA capacity bumped: `CFG.sd_lora_r` 8 → 16, `CFG.sd_lora_alpha` 16 → 32. Rationale: at r=8 the projector + UNet-LoRA was under-fitting attribute-transformation edits (e.g. "hair → blonde" producing a flat color blob instead of textured hair). r=16 roughly doubles the trainable LoRA capacity in the targeted attention modules.
- Training horizon extended: `CFG.phase2_epochs` 5 → 9. Loss was still descending at epoch 5 in the v1.6 run, and the bigger LoRA needs more steps to converge. The cosine LR schedule is re-derived from `phase2_epochs` at function-call time, so this also re-fits the cosine to the new horizon (no near-zero-LR tail).
- `CFG.ckpt_phase2` is now also keyed on the LoRA rank, not just the cache version. The path becomes `checkpoints/phase2_diffusion{cache_suffix}_r{lora_r}/`. This is critical: LoRA rank changes the shape of the adapter weight tensors, so an r=8 checkpoint cannot be loaded into an r=16 model. Keeping them in separate dirs makes the §18.2 fresh-start guard catch this automatically and keeps the old r=8 v1.6 run intact for comparison.

**How to run this version**

1. Leave `CFG.vlm_cache_version` at `"v2"` (the existing hidden-state cache is reused — VLM hidden states are independent of UNet LoRA rank).
2. Confirm the new `ckpt_phase2` dir is empty (`checkpoints/phase2_diffusion_v2_r16/`). The §18.2 guard will hard-abort if it isn't.
3. Run §18.2 from scratch — do **not** pass `resume_from`. The r=8 checkpoint is incompatible with the r=16 model, and the cache-version gate alone would not catch this (hence the rank-keyed path).
4. Expect ~9/5 × the v1.6 wallclock per epoch baseline, plus a small overhead from the larger LoRA — budget accordingly on A100.

**What did not change**

- VLM hidden-state cache, projection adapter dimensionality, prompt template, mask paths (Phase 2 = GT RLE, inference = VLM bbox → SAM2), 9-channel inpainting UNet input, CFG handling. All §0.0a (v1.6) safety guards remain in force.


## §0.0a — v1.6 changelog (history)

**What changed vs. v1.5**

- `CFG.vlm_cache_version` toggles the VLM hidden-state cache. Set to `"v2"` to point at `vlm_hidden_states_v2/` and `samples_filtered_v2.json`.
- `CFG.ckpt_phase2` is now per-cache-version (`phase2_diffusion_v2/`) so a v1-trained projector + UNet-LoRA cannot be silently resumed against v2 hidden states.
- `_save_phase2_checkpoint` now persists optimizer / scheduler / scaler / RNG state, and `run_phase2_training(resume_from=...)` restores them. Mid-run resumes now actually continue the cosine LR curve and Adam moments instead of restarting them.
- §18.2 hard-aborts if `ckpt_phase2` is non-empty and `RESUME_FROM` is `None` — no silent train-train mismatch.

**When you re-train the VLM**

1. Re-extract hidden states with the *same* §8.1 prompt template into `vlm_hidden_states_v2/` (the §13.1 template-hash gate enforces this).
2. Bump `CFG.vlm_cache_version` here.
3. Run §13.1 — confirm template hash matches, shard coverage ≥50%, and the new ckpt dir is empty.
4. Run §18.2 from scratch (do not pass `resume_from`). The projector + UNet-LoRA must be re-fit to the new VLM's hidden-state distribution.


## §0 — Global Configuration
*Identical base to Phase 0/1; Phase 2 fields appended. Edit this cell only.*

In [ ]:
# ── §0.1  GLOBAL CONFIG — edit this cell only ──────────────────────────────
# All downstream cells import from CFG; never hardcode paths elsewhere.
# Phase 2 additions are marked with: # ← Phase 2

import os, sys, json
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

@dataclass
class PipelineConfig:
    # ── Drive root ────────────────────────────────────────────────────────
    drive_root: Path = Path("/content/drive/MyDrive/img_edit_pipeline")

    # ── Dataset ───────────────────────────────────────────────────────────
    hf_dataset_id: str         = "sysuyy/ImgEdit"
    hf_configs: Optional[list] = None
    n_subset: int              = 10_000

    # Confirmed singleturn edit types (Phase 0 §2.1 — expanded beyond spec)
    expected_edit_types: tuple = (
        "action", "add", "adjust", "background", "content",
        "hybrid", "reference", "remove", "replace", "style", "version",
    )
    mask_dataset_id: str = "sysuyy/ImgEdit_recap_mask"

    global_adjust_keywords: tuple = (
        "lighting", "light", "brightness", "exposure", "contrast",
        "saturation", "hue", "tone", "tones", "overall", "scene",
        "atmosphere", "color temperature", "white balance",
    )

    # ── Phase 1 model ─────────────────────────────────────────────────────
    vlm_model_id: str       = "Qwen/Qwen2.5-VL-3B-Instruct"
    vlm_quant_bits: int     = 4
    vlm_lora_r: int         = 16
    vlm_lora_alpha: int     = 32
    vlm_lora_dropout: float = 0.05
    vlm_hidden_dim: int     = 2048   # Qwen2.5-VL-3B confirmed hidden size

    # ── Phase 2 model ─────────────────────────────────────────────────────
    sd_model_id: str        = "runwayml/stable-diffusion-inpainting"
    sd_cross_attn_dim: int  = 768    # SD 1.5 cross-attention dim (constant)
    sd_lora_r: int          = 16
    sd_lora_alpha: int      = 32
    sd_lora_dropout: float  = 0.0    # ← Phase 2

    # UNet LoRA target modules — verified in §14.2 by printing module names.
    # Standard SD 1.5 attention projection names (to_q/k/v/out.0).
    sd_lora_target_modules: tuple = ("to_q", "to_k", "to_v", "to_out.0")  # ← Phase 2

    # VAE latent scaling factor (SD 1.5 constant — do not change)
    vae_scale_factor: float = 0.18215   # ← Phase 2

    # ── VLM hidden-state cache version (v1.6) ────────────────────────────
    # When the VLM is re-trained for more epochs, re-extract hidden states
    # into a NEW directory (vlm_hidden_states_v2) and bump this string.
    # Path properties below switch on this value so v1 and v2 caches
    # never collide. The Phase 2 checkpoint dir is also versioned, so
    # a v1-trained projector cannot be silently resumed against v2 hidden
    # states (the projector + UNet-LoRA were fit to the v1 distribution).
    #
    # Set to "v1" for the original cache, "v2" for the 3-epoch-VLM cache.
    vlm_cache_version: str = "v2"   # ← Phase 2 (v1.6)

    # Training resolution — images resized to this before VAE encode
    phase2_resolution: int  = 512       # ← Phase 2

    # ── Training ──────────────────────────────────────────────────────────
    phase1_epochs: int         = 3
    phase1_batch_size: int     = 2
    phase1_grad_accum: int     = 8
    phase1_lr: float           = 2e-4
    phase1_max_seq_len: int    = 2560
    phase1_max_img_pixels: int = 802_816
    phase1_warmup_steps: int   = 100
    phase1_eval_steps: int     = 200

    phase2_epochs: int         = 9
    phase2_batch_size: int     = 8
    phase2_grad_accum: int     = 4
    phase2_lr: float           = 1e-4
    phase2_warmup_steps: int   = 150
    phase2_shard_size: int     = 5_000
    phase2_max_grad_norm: float = 1.0   # ← Phase 2
    phase2_num_workers: int    = 2      # ← Phase 2: DataLoader workers
    phase2_save_steps: int     = 500    # ← Phase 2: checkpoint every N steps
    phase2_mixed_precision: str = "fp16" # ← Phase 2: "fp16" or "bf16"

    # Noise scheduler (DDPM for training; SD 1.5 defaults)
    phase2_num_train_timesteps: int = 1000  # ← Phase 2
    phase2_beta_schedule: str       = "linear"  # ← Phase 2

    # ── Audit thresholds ──────────────────────────────────────────────────
    max_invalid_rle_frac: float = 0.05
    max_fallback_frac: float    = 0.15

    # ── Derived paths ─────────────────────────────────────────────────────
    @property
    def data_dir(self) -> Path:
        return self.drive_root / "data" / "imgedit_subset"

    @property
    def benchmark_dir(self) -> Path:
        return self.drive_root / "data" / "benchmark"

    @property
    def ckpt_phase1(self) -> Path:
        return self.drive_root / "checkpoints" / "phase1_vlm"

    @property
    def ckpt_phase2(self) -> Path:
        # v1.7: per-cache-version AND per-LoRA-rank checkpoint dir.
        # v1 (r=8)  → checkpoints/phase2_diffusion              (legacy, untouched)
        # v2 (r=8)  → checkpoints/phase2_diffusion_v2           (legacy v1.6 run)
        # v2 (r=16) → checkpoints/phase2_diffusion_v2_r16       (fresh v1.7 run)
        # The rank suffix is critical: changing sd_lora_r changes the
        # LoRA tensor shapes, so an r=8 checkpoint cannot be loaded
        # into an r=16 model. Keying the path on rank means the §18.2
        # fresh-start guard catches this automatically.
        _legacy = self._cache_suffix == "" and self.sd_lora_r == 8  # untouch v1 path
        _rank_suffix = "" if _legacy else f"_r{self.sd_lora_r}"
        return self.drive_root / "checkpoints" / f"phase2_diffusion{self._cache_suffix}{_rank_suffix}"

    @property
    def outputs_eval(self) -> Path:
        return self.drive_root / "outputs" / "eval"

    @property
    def outputs_scores(self) -> Path:
        return self.drive_root / "outputs" / "scores"

    @property
    def manifest_path(self) -> Path:
        return self.data_dir / "samples.json"

    @property
    def _cache_suffix(self) -> str:
        # "" for v1 (original layout), "_v2" for the 3-epoch-VLM cache, etc.
        v = self.vlm_cache_version
        return "" if v in ("v1", "", None) else f"_{v}"

    @property
    def filtered_manifest_path(self) -> Path:
        # v1.6: switches on vlm_cache_version
        return self.data_dir / f"samples_filtered{self._cache_suffix}.json"

    @property
    def hidden_states_dir(self) -> Path:
        # v1.6: switches on vlm_cache_version
        return self.data_dir / f"vlm_hidden_states{self._cache_suffix}"

    @property
    def hidden_states_meta_path(self) -> Path:
        return self.hidden_states_dir / "hidden_states_meta.json"

    @property
    def audit_path(self) -> Path:
        return self.data_dir / "audit_report.json"


CFG = PipelineConfig()
print("Config loaded.")
print(f"  Drive root        : {CFG.drive_root}")
print(f"  SD model          : {CFG.sd_model_id}")
print(f"  VLM hidden dim    : {CFG.vlm_hidden_dim}")
print(f"  SD cross-attn dim : {CFG.sd_cross_attn_dim}")
print(f"  Phase 2 resolution: {CFG.phase2_resolution}")
print(f"  Phase 2 epochs    : {CFG.phase2_epochs}")
print(f"  Phase 2 lr        : {CFG.phase2_lr}")
print(f"  VLM cache version : {CFG.vlm_cache_version}")
print(f"  Filtered manifest : {CFG.filtered_manifest_path}")
print(f"  Hidden states dir : {CFG.hidden_states_dir}")
print(f"  Phase 2 ckpt      : {CFG.ckpt_phase2}")


Config loaded.
  Drive root        : /content/drive/MyDrive/img_edit_pipeline
  SD model          : runwayml/stable-diffusion-inpainting
  VLM hidden dim    : 2048
  SD cross-attn dim : 768
  Phase 2 resolution: 512
  Phase 2 epochs    : 9
  Phase 2 lr        : 0.0001
  VLM cache version : v2
  Filtered manifest : /content/drive/MyDrive/img_edit_pipeline/data/imgedit_subset/samples_filtered_v2.json
  Hidden states dir : /content/drive/MyDrive/img_edit_pipeline/data/imgedit_subset/vlm_hidden_states_v2
  Phase 2 ckpt      : /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16


## §0.2 — Google Drive Mount & Directory Scaffold

In [ ]:
# ── §0.2  Mount Drive and create Phase 2 directories ─────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Create all required directories
_dirs = [
    CFG.data_dir,
    CFG.hidden_states_dir,
    CFG.ckpt_phase2,
    CFG.outputs_eval,
    CFG.outputs_scores,
]
for d in _dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  ok  {d}")

print("\nDirectory scaffold verified.")
print(f"  Phase 2 checkpoint dir : {CFG.ckpt_phase2}")


Mounted at /content/drive
  ok  /content/drive/MyDrive/img_edit_pipeline/data/imgedit_subset
  ok  /content/drive/MyDrive/img_edit_pipeline/data/imgedit_subset/vlm_hidden_states_v2
  ok  /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16
  ok  /content/drive/MyDrive/img_edit_pipeline/outputs/eval
  ok  /content/drive/MyDrive/img_edit_pipeline/outputs/scores

Directory scaffold verified.
  Phase 2 checkpoint dir : /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16


## §1.1 — Package Installation
⚠ **Run once per runtime.** After this cell, restart the kernel then continue from §1.2.

These are the same pinned versions as Phase 0/1 plus Phase 2 extras.
Re-running is safe — pip silently skips already-installed packages at the right version.


In [ ]:
# ── §1.1  Install pinned dependencies ────────────────────────────────────
# Colab's runtime changes frequently. Strategy:
#   - torch/torchvision: NOT reinstalled — use whatever Colab provides (2.2+)
#   - numpy: pin <2.0 FIRST (ABI break with pycocotools + torch C-extensions)
#   - everything else: minimum version pins (>= not ==) so Colab upgrades don't block
#
# After this cell finishes: RESTART KERNEL, then run §1.2.

import subprocess, sys

def _install(pkg, label=None):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    tag = label or pkg
    if r.returncode == 0:
        print(f"  ok  {tag}")
    else:
        print(f"  FAILED  {tag}")
        print(r.stderr[-400:])

# ── Step 1: numpy<2 FIRST ─────────────────────────────────────────────────
# pycocotools and some torch C-extensions were compiled against NumPy 1.x.
# NumPy 2.x has a breaking ABI change — downgrade before anything else runs.
print("Step 1: numpy<2 (must come before all other installs)...")
_install("numpy<2", "numpy<2")

# ── Step 2: Phase 2 packages ──────────────────────────────────────────────
# Use >= pins so Colab runtime updates (e.g. torch 2.10, Pillow 11) don't block.
# The SD inpainting APIs (AutoencoderKL, UNet2DConditionModel, DDPMScheduler,
# StableDiffusionInpaintPipeline) have been stable in diffusers since 0.27.
print("\nStep 2: Phase 2 packages...")
_pkgs = [
    ("diffusers>=0.27.0",    "diffusers>=0.27"),
    ("peft>=0.10.0",         "peft>=0.10"),
    ("accelerate>=0.28.0",   "accelerate>=0.28"),
    ("transformers>=4.49.0", "transformers>=4.49"),
    ("pycocotools>=2.0.7",   "pycocotools>=2.0.7"),
    ("tqdm>=4.66",           "tqdm>=4.66"),
    ("Pillow>=10.0",         "Pillow>=10"),
]
for pkg, label in _pkgs:
    _install(pkg, label)

# ── Step 3: torchao version fix (required for PEFT LoRA) ─────────────────
# Root cause: PEFT's dispatch_torchao dispatcher raises ImportError when
# torchao is installed at < 0.16.0. Colab ships torchao 0.10.0 by default,
# which triggers this error when get_peft_model() is called in §14.4.
#
# Fix: upgrade torchao to satisfy PEFT's minimum. torchao is an orthogonal
# optimization library; upgrading it does NOT affect torch, diffusers, or
# any other Phase 2 dependency.
#
# If torchao is not installed at all, PEFT simply skips the torchao
# dispatcher and uses the standard PyTorch Linear replacement — no issue.
print("\nStep 3: torchao >= 0.16.0 (PEFT LoRA compatibility fix)...")
_install("torchao>=0.16.0", "torchao>=0.16.0")

print()
print("Installation complete.")
print("⚠  RESTART KERNEL NOW, then run §1.2.")
print("   Runtime → Restart session  (Ctrl+M .)  then continue from §1.2.")


Step 1: numpy<2 (must come before all other installs)...
  ok  numpy<2

Step 2: Phase 2 packages...
  ok  diffusers>=0.27
  ok  peft>=0.10
  ok  accelerate>=0.28
  ok  transformers>=4.49
  ok  pycocotools>=2.0.7
  ok  tqdm>=4.66
  ok  Pillow>=10

Step 3: torchao >= 0.16.0 (PEFT LoRA compatibility fix)...
  ok  torchao>=0.16.0

Installation complete.
⚠  RESTART KERNEL NOW, then run §1.2.
   Runtime → Restart session  (Ctrl+M .)  then continue from §1.2.


## §1.2 — Import & Version Sanity Checks
*Run after every kernel restart. All assertions must pass before any Phase 2 cell runs.*

In [ ]:
# ── §1.2  Import and version gate ────────────────────────────────────────
# Checks minimum required versions rather than exact versions so Colab
# runtime upgrades (torch 2.10, Pillow 11, diffusers 0.30+) do not block.
#
# Hard requirements:
#   numpy  < 2.0   — NumPy 2.x breaks pycocotools ABI
#   torch  >= 2.2  — minimum for mixed precision + PEFT LoRA on UNet
#   diffusers >= 0.27 — SD inpainting component API (stable since 0.27)
#   peft >= 0.10   — LoraConfig + get_peft_model
#   accelerate >= 0.28 — GradScaler
#   transformers >= 4.49 — Qwen2.5-VL support (used in Phase 1/3)

import importlib, importlib.metadata, sys
from pathlib import Path

_failures = []

def _ver(pkg_name: str) -> str:
    """Get installed version string, or '0.0.0' if not found."""
    try:
        return importlib.metadata.version(pkg_name)
    except importlib.metadata.PackageNotFoundError:
        return None

def _parse(ver_str: str) -> tuple:
    """Parse 'X.Y.Z+...' → (X, Y, Z) int tuple for comparison."""
    clean = ver_str.split("+")[0].split("-")[0]   # strip +cu128, -rc1 etc.
    parts = clean.split(".")
    nums  = []
    for p in parts[:3]:
        try:    nums.append(int(p))
        except: nums.append(0)
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

def _check(pip_name: str, min_ver: str, max_ver: str = None, mod_name: str = None):
    """Verify pip_name is installed and min_ver <= installed < max_ver."""
    ver = _ver(pip_name)
    if ver is None:
        # Try importing as a fallback (e.g. PIL → Pillow)
        if mod_name:
            try:
                m = importlib.import_module(mod_name)
                ver = getattr(m, "__version__", None)
            except ImportError:
                pass
    if ver is None:
        _failures.append(f"  MISSING: {pip_name} — run §1.1 and restart kernel")
        return
    inst  = _parse(ver)
    minv  = _parse(min_ver)
    ok    = inst >= minv
    if max_ver:
        ok = ok and inst < _parse(max_ver)
    if ok:
        print(f"  ok  {pip_name}=={ver}")
    else:
        constraint = f">= {min_ver}" + (f", < {max_ver}" if max_ver else "")
        _failures.append(f"  WRONG VERSION: {pip_name}=={ver} (need {constraint})")

# numpy must be < 2.0 (hard upper bound — ABI break)
_check("numpy",        min_ver="1.0.0", max_ver="2.0.0")
# torch >= 2.2 (need GradScaler + PEFT UNet LoRA support)
_check("torch",        min_ver="2.2.0", mod_name="torch")
_check("diffusers",    min_ver="0.27.0")
_check("peft",         min_ver="0.10.0")
_check("accelerate",   min_ver="0.28.0")
_check("transformers", min_ver="4.49.0")
_check("Pillow",       min_ver="10.0.0", mod_name="PIL")
_check("pycocotools",  min_ver="2.0.0")

# torchao: if installed, must be >= 0.16.0 (PEFT LoRA requirement)
# Not a hard required package — absence is fine; old version is not.
_tao_ver = _ver("torchao")
if _tao_ver is not None:
    _check("torchao", min_ver="0.16.0")
else:
    print("  ok  torchao (not installed — PEFT will use standard dispatcher)")


if _failures:
    raise RuntimeError(
        "Version gate failed:\n" + "\n".join(_failures) +
        "\n\nFixes:\n"
        "  numpy wrong  : pip install \'numpy<2\' then RESTART KERNEL\n"
        "  pkg missing  : run §1.1, then RESTART KERNEL, then re-run §1.2\n"
        "  torch wrong  : Colab runtime must have torch>=2.2 — check Runtime type"
    )

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device found. "
        "Switch to Runtime → Change runtime type → GPU (T4 minimum, A100 preferred)."
    )

_device_name = torch.cuda.get_device_name(0)
_vram_gb     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n  CUDA device  : {_device_name}")
print(f"  VRAM         : {_vram_gb:.1f} GB")
if _vram_gb < 15:
    print("  WARNING: <15 GB VRAM. Phase 2 training needs ≥16 GB (T4 min, A100 preferred).")

DEVICE = "cuda"
print(f"  DEVICE       : {DEVICE}")
print("\n§1.2 passed — all imports and versions OK.")


  ok  numpy==1.26.4
  ok  torch==2.10.0+cu128
  ok  diffusers==0.37.1
  ok  peft==0.19.1
  ok  accelerate==1.13.0
  ok  transformers==5.0.0
  ok  Pillow==11.3.0
  ok  pycocotools==2.0.11
  ok  torchao==0.17.0

  CUDA device  : NVIDIA A100-SXM4-40GB
  VRAM         : 42.4 GB
  DEVICE       : cuda

§1.2 passed — all imports and versions OK.


## §3.1 — Core Utilities (Phase 2 Subset)
Includes `decode_rle_mask`, `select_target_bbox`, `select_best_annotation`, `get_training_mask`, bbox helpers, `load_json`, `save_json`.

⚠ `select_target_bbox` is byte-identical to Phase 1 §3.1 — do not modify.

In [ ]:
# ── §3.1  Core utilities — Phase 2 subset ────────────────────────────────
# Included from Phase 0/1 §3.1:
#   decode_rle_mask, select_target_bbox, bbox helpers, load_json, save_json
#
# CRITICAL: select_target_bbox and GLOBAL_ADJUST_KEYWORDS must be byte-identical
# to Phase 1 §3.1. Any divergence means Phase 2 picks different masks than the
# VLM was trained to reference — a train/inference mismatch for the adapter.

import io, json, logging
import numpy as np
from pathlib import Path
from typing import Any, Optional
from PIL import Image

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler()],
)
log = logging.getLogger("pipeline")

import pycocotools.mask as mask_utils

def decode_rle_mask(rle: dict) -> Optional[np.ndarray]:
    """Decode COCO-format RLE dict → binary uint8 mask (H×W). Returns None on failure."""
    if rle is None:
        return None
    try:
        rle_copy = dict(rle)
        if isinstance(rle_copy.get("counts"), str):
            rle_copy["counts"] = rle_copy["counts"].encode("utf-8")
        mask = mask_utils.decode(rle_copy)
        assert mask.ndim == 2, f"Expected 2-D mask, got {mask.shape}"
        return mask
    except Exception as e:
        log.debug(f"RLE decode failed: {e}")
        return None


KNOWN_EDIT_TYPES = set(CFG.expected_edit_types)
GLOBAL_ADJUST_KEYWORDS = set(CFG.global_adjust_keywords)


def bbox_area_absolute(bbox_xyxy: list) -> float:
    x1, y1, x2, y2 = bbox_xyxy
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def clip_bbox_to_image(bbox_xyxy: list, W: int, H: int) -> list:
    x1, y1, x2, y2 = bbox_xyxy
    return [max(0, min(x1, W)), max(0, min(y1, H)),
            max(0, min(x2, W)), max(0, min(y2, H))]

def bbox_to_relative(bbox_xyxy: list, W: int, H: int) -> list:
    x1, y1, x2, y2 = clip_bbox_to_image(bbox_xyxy, W, H)
    return [round(x1/W*1000), round(y1/H*1000), round(x2/W*1000), round(y2/H*1000)]

def bbox_relative_to_absolute(bbox_rel: list, W: int, H: int) -> list:
    x1, y1, x2, y2 = bbox_rel
    return [round(x1/1000*W), round(y1/1000*H), round(x2/1000*W), round(y2/1000*H)]


def select_target_bbox(
    edit_type: str,
    edit_description: str,
    annotations: list,
    img_W: int,
    img_H: int,
) -> tuple:
    """Select ground-truth target bbox. IDENTICAL to Phase 1 §3.1 — DO NOT MODIFY.

    Returns (bbox_rel [0,1000], matched_by_str).
    Phase 2 uses this to pick the matching annotation for mask extraction —
    the same annotation the VLM was trained to reference.
    """
    FULL_IMAGE = [0, 0, 1000, 1000]
    et = (edit_type or "").strip().lower()

    if et in {"style", "background"}:
        return FULL_IMAGE, "global"
    if et == "adjust":
        desc_lower = (edit_description or "").lower()
        if any(kw in desc_lower for kw in GLOBAL_ADJUST_KEYWORDS):
            return FULL_IMAGE, "global"

    instruction_lower = (edit_description or "").lower()
    all_anns = list(annotations or [])

    if not all_anns:
        return FULL_IMAGE, "area_fallback"

    # Step 1: class-name substring matching
    matched_anns = []
    for ann in all_anns:
        class_name = (ann.get("class_name") or "").strip().lower()
        if class_name and class_name in instruction_lower:
            matched_anns.append(ann)

    if matched_anns:
        if len(matched_anns) == 1:
            best_ann, matched_by = matched_anns[0], "class_name"
        else:
            # Step 2: clip_score tiebreaker, then area
            scored = []
            for ann in matched_anns:
                cs   = ann.get("clip_score")
                area = bbox_area_absolute(ann.get("bbox") or []) if len(ann.get("bbox") or []) == 4 else 0.0
                scored.append((cs is not None, cs or 0.0, area, ann))
            scored.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
            best_ann   = scored[0][3]
            matched_by = "clip_score" if scored[0][0] else "class_name"

        raw_bbox = best_ann.get("bbox") or []
        if len(raw_bbox) == 4 and bbox_area_absolute(raw_bbox) > 0:
            return bbox_to_relative(raw_bbox, img_W, img_H), matched_by

    # Step 3: area fallback
    log.debug(f"select_target_bbox: no class match in {instruction_lower[:60]!r} — area fallback")
    object_anns    = [a for a in all_anns if a.get("region") == "object"]
    candidate_anns = object_anns if object_anns else all_anns
    best_bbox, best_area = None, 0.0
    for ann in candidate_anns:
        raw_bbox = ann.get("bbox")
        if raw_bbox is None or len(raw_bbox) != 4:
            continue
        area = bbox_area_absolute(raw_bbox)
        if area > best_area:
            best_area, best_bbox = area, raw_bbox
    if best_bbox is None or best_area == 0.0:
        return FULL_IMAGE, "area_fallback"
    return bbox_to_relative(best_bbox, img_W, img_H), "area_fallback"


def select_best_annotation(
    edit_type: str,
    edit_description: str,
    annotations: list,
) -> Optional[dict]:
    """Return the full annotation dict selected by select_target_bbox logic.

    Phase 2 uses this to extract the ground-truth RLE mask for inpainting
    supervision. Returns None if no valid annotation (global edit or empty list).

    NOTE: For global edits (style/background/global-adjust), returns None.
    The caller should use a full-image mask (all-ones) in that case.
    This function does NOT return a fallback annotation for global edits —
    a full-image mask trained from a specific annotation would be wrong.
    """
    et = (edit_type or "").strip().lower()
    if et in {"style", "background"}:
        return None
    if et == "adjust":
        desc_lower = (edit_description or "").lower()
        if any(kw in desc_lower for kw in GLOBAL_ADJUST_KEYWORDS):
            return None

    instruction_lower = (edit_description or "").lower()
    all_anns = list(annotations or [])
    if not all_anns:
        return None

    # Class-name match
    matched = [a for a in all_anns
               if (a.get("class_name") or "").strip().lower()
               and (a.get("class_name") or "").strip().lower() in instruction_lower]
    if matched:
        if len(matched) == 1:
            return matched[0]
        scored = []
        for ann in matched:
            cs   = ann.get("clip_score")
            area = bbox_area_absolute(ann.get("bbox") or []) if len(ann.get("bbox") or []) == 4 else 0.0
            scored.append((cs is not None, cs or 0.0, area, ann))
        scored.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
        return scored[0][3]

    # Area fallback
    object_anns    = [a for a in all_anns if a.get("region") == "object"]
    candidate_anns = object_anns if object_anns else all_anns
    best_ann, best_area = None, 0.0
    for ann in candidate_anns:
        raw_bbox = ann.get("bbox")
        if raw_bbox is None or len(raw_bbox) != 4:
            continue
        area = bbox_area_absolute(raw_bbox)
        if area > best_area:
            best_area, best_ann = area, ann
    return best_ann  # None if still not found


def get_training_mask(
    edit_type: str,
    instruction: str,
    annotations: list,
    target_H: int,
    target_W: int,
) -> np.ndarray:
    """Get binary uint8 training mask (H×W) from ground-truth RLE annotations.

    Phase 2 ONLY — uses RLE from segmentation JSON (NOT SAM2).
    Phase 3 inference uses SAM2-generated masks from VLM bbox predictions.

    Mask polarity: 1 = region to inpaint (edit here), 0 = region to keep.
    For global edits (no annotation match): returns all-ones (full-image mask).

    Args:
        edit_type:   from manifest entry
        instruction: natural-language edit instruction
        annotations: list of annotation dicts from the segmentation JSON.
                     Each dict has "segmentation": {counts, size} (COCO RLE).
        target_H/W:  desired output mask size (image resolution after resize).

    Returns:
        np.ndarray, dtype=uint8, shape=(target_H, target_W), values in {0, 1}.
    """
    ann = select_best_annotation(edit_type, instruction, annotations)

    if ann is None:
        # Global edit or no annotation found: full-image mask
        return np.ones((target_H, target_W), dtype=np.uint8)

    rle_dict = ann.get("segmentation")
    if rle_dict is None:
        log.debug(f"Annotation has no segmentation field — using full-image mask")
        return np.ones((target_H, target_W), dtype=np.uint8)

    mask = decode_rle_mask(rle_dict)
    if mask is None or mask.sum() == 0:
        log.debug(f"RLE decode failed or empty mask — using full-image mask")
        return np.ones((target_H, target_W), dtype=np.uint8)

    # Resize mask to target image size if dimensions differ
    # Uses NEAREST interpolation to preserve binary values (no aliasing)
    if mask.shape != (target_H, target_W):
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8)).convert("L")  # mode= kwarg deprecated in Pillow 13
        mask_pil = mask_pil.resize((target_W, target_H), Image.NEAREST)
        mask = (np.array(mask_pil) > 127).astype(np.uint8)

    assert mask.shape == (target_H, target_W), (
        f"get_training_mask: output shape {mask.shape} != ({target_H},{target_W})"
    )
    assert mask.dtype == np.uint8
    return mask


def save_json(obj: Any, path: Path, indent: int = 2) -> None:
    """Atomic JSON write (tmp → rename)."""
    path = Path(path)
    tmp  = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=indent, ensure_ascii=False))
    tmp.rename(path)

def load_json(path: Path) -> Any:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Expected JSON not found: {path}")
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError as e:
        raise ValueError(f"Corrupt JSON at {path}: {e}") from e


# ── Smoke test ──────────────────────────────────────────────────────────────
_test_anns = [
    {"class_name": "shirt",  "bbox": [300, 400, 700, 900], "region": "object",
     "clip_score": 0.82, "segmentation": None},
    {"class_name": "person", "bbox": [100, 50,  900, 1200], "region": "object",
     "clip_score": 0.61, "segmentation": None},
]
_bbox, _by = select_target_bbox("adjust", "change the shirt to blue", _test_anns, 1920, 1080)
assert _by == "class_name", f"Expected class_name, got {_by}"

_ann_sel = select_best_annotation("adjust", "change the shirt to blue", _test_anns)
assert _ann_sel is not None and _ann_sel["class_name"] == "shirt", "select_best_annotation failed"

_mask = get_training_mask("style", "make it more vivid", [], 64, 64)
assert _mask.shape == (64, 64) and _mask.all(), "Global edit should give all-ones mask"

print("§3.1 utilities loaded (Phase 2 subset):")
print("  decode_rle_mask, select_target_bbox, select_best_annotation")
print("  get_training_mask, bbox helpers, load_json, save_json")
print("  ok  select_target_bbox smoke test passed")
print("  ok  get_training_mask global-edit smoke test passed")


§3.1 utilities loaded (Phase 2 subset):
  decode_rle_mask, select_target_bbox, select_best_annotation
  get_training_mask, bbox helpers, load_json, save_json
  ok  select_target_bbox smoke test passed
  ok  get_training_mask global-edit smoke test passed


## §8.1 — Phase 1 Prompt Template (Read-Only Reference)
Copied verbatim from Phase 1 §8.1. Used only to verify `PHASE1_TEMPLATE_HASH` against cached hidden states. Not called during Phase 2 training.

In [ ]:
# ── §8.1  Phase 1 prompt template constants — IMPORTED VERBATIM ──────────
#
# CRITICAL: These constants MUST be byte-identical to Phase 1 §8.1.
# Phase 2 uses them to validate that hidden states were extracted with the
# same template. ANY change here invalidates the cached hidden states.
#
# Phase 2 does NOT call build_messages() for training — hidden states are
# pre-cached. This block is included so Phase 2 can:
#   1. Verify PHASE1_TEMPLATE_HASH against hidden_states_meta.json
#   2. Provide the exact prompt template to Phase 3 via import
#
# format_regions_text and build_messages are defined here for completeness
# and Phase 3 reference. They are NOT called during Phase 2 training.

import hashlib as _hashlib
import json as _json_mod

PHASE1_SYSTEM_PROMPT = (
    "You are an image editing assistant. "
    "Given a source image, a list of segmentation regions with bounding boxes, "
    "and a natural-language edit instruction, "
    "identify the target region and predict the edit operation as a JSON object.\n\n"
    "The segmentation regions are listed as:\n"
    "  - <class_name>: [x1, y1, x2, y2]\n"
    "where coordinates are in [0, 1000] relative scale "
    "(0 = top-left, 1000 = bottom-right).\n\n"
    "JSON schema:\n"
    "{\n"
    '  "edit_type": "<type>",          '
    "// one of: action, add, adjust, background, content,\n"
    "//         hybrid, reference, remove, replace, style, version\n"
    '  "bbox": [x1, y1, x2, y2],       '
    "// the bounding box of the target region in [0, 1000] coordinates\n"
    "//   (select the region from the list above that best matches the instruction)\n"
    '  "edit_description": "<text>"    '
    "// precise description of what to edit\n"
    "}\n\n"
    "Output ONLY valid JSON. No explanation, no markdown code fences."
)

PHASE1_USER_REGIONS_HEADER = "Segmentation regions:\n"
PHASE1_USER_PREFIX          = "Edit instruction: "

_tpl_str = PHASE1_SYSTEM_PROMPT + "|SEP|" + PHASE1_USER_REGIONS_HEADER + "|SEP|" + PHASE1_USER_PREFIX
PHASE1_TEMPLATE_HASH = _hashlib.sha256(_tpl_str.encode()).hexdigest()[:16]
print(f"PHASE1_TEMPLATE_HASH = {PHASE1_TEMPLATE_HASH!r}")
print("  This must match hidden_states_meta.json['phase1_template_hash'] (verified in §13.1).")


PHASE1_TEMPLATE_HASH = 'e8964e0ced3f9891'
  This must match hidden_states_meta.json['phase1_template_hash'] (verified in §13.1).


## §13.1 — Phase 2 Pre-Flight Checks
**Gate:** all checks must pass before §14 (model loading). Validates manifest, shard coverage, template hash, shard shapes, and image files.

In [ ]:
# ── §13.1  Validate Phase 1 artifacts before any model loading ──────────
#
# GATE: All checks must pass before §14 (model loading) runs.
#
# NOTE on missing shard_id:
#   Not every manifest entry is expected to have a shard_id. Samples can
#   legitimately have no shard_id for two distinct reasons:
#
#   (A) Phase 1 tokenization failure (expected, permanent):
#       The sample exceeded phase1_max_seq_len, had a corrupt/missing image,
#       or produced an empty response after the chat template was applied.
#       These samples were correctly skipped by Phase 1 §9.4 and §11.
#       They will never get a shard_id. Phase2Dataset already handles this
#       by filtering them out at init time.
#
#   (B) Incomplete extraction run (fixable, temporary):
#       Phase 1 §11.3 was interrupted before finishing. The manifest has
#       entries waiting for extraction. Re-running §11.3 (restart-safe)
#       will fill in the remaining shard_ids.
#
#   This cell distinguishes (A) from (B) using a hard lower bound:
#   if fewer than MIN_SHARD_COVERAGE (50%) of samples have shard_id,
#   it is almost certainly (B) — extraction never ran or was badly interrupted.
#   Otherwise we treat missing entries as (A) and log a warning.
#
# Checks:
#   1. samples_filtered.json exists and is non-empty
#   2. At least MIN_SHARD_COVERAGE of samples have shard_id
#   3. hidden_states_meta.json exists with matching PHASE1_TEMPLATE_HASH
#   4. At least one shard_*.pt file exists and loads correctly
#   5. Shard round-trip: manifest -> shard -> hidden_state shape check
#   6. Source image and segmentation JSON spot-check on a random sample

import torch, random

# Minimum fraction of samples that must have shard_id for training to proceed.
# Samples below this threshold indicate extraction never ran (not just filtering).
# Set to 50%: if fewer than half have shards, something went wrong upstream.
MIN_SHARD_COVERAGE = 0.50

print("=" * 60)
print("§13.1  Phase 1 artifact validation")
print("=" * 60)

# v1.6: announce which VLM hidden-state cache version this run targets.
# The CFG path properties switch on CFG.vlm_cache_version, so set that in §0.1.
print(f"  cache version    : {CFG.vlm_cache_version!r}")
print(f"  manifest path    : {CFG.filtered_manifest_path}")
print(f"  hidden-states dir: {CFG.hidden_states_dir}")
print(f"  ckpt dir         : {CFG.ckpt_phase2}")

# v1.6 safety check: warn loudly if the chosen ckpt_phase2 dir already
# contains checkpoints. Resuming an old projector + UNet-LoRA against a
# newly-extracted hidden-state cache is a silent train-train mismatch
# (the projector was fit to a different VLM checkpoint's distribution).
if CFG.ckpt_phase2.exists() and any(CFG.ckpt_phase2.iterdir()):
    _existing = sorted(p.name for p in CFG.ckpt_phase2.iterdir())
    print(
        f"  WARNING: {CFG.ckpt_phase2} is non-empty: {_existing[:5]}"
        + ("..." if len(_existing) > 5 else "")
    )
    print(
        "           If you just bumped vlm_cache_version, this directory\n"
        "           should be empty before the first run. Old checkpoints\n"
        "           were trained on a DIFFERENT VLM hidden-state distribution\n"
        "           and resuming them now is a train-train mismatch.\n"
        "           Move/delete them OR pass an explicit resume_from= path."
    )

# ── 1. Manifest ───────────────────────────────────────────────────────────
assert CFG.filtered_manifest_path.exists(), (
    f"samples_filtered.json not found: {CFG.filtered_manifest_path}\n"
    "Run Phase 0 §7.1 (full download) and §7.3 (filter) before Phase 2."
)
_manifest = load_json(CFG.filtered_manifest_path)
assert len(_manifest) > 0, "samples_filtered.json is empty — re-run Phase 0 §7.3."
print(f"  ok  manifest: {len(_manifest):,} samples from {CFG.filtered_manifest_path.name}")

# ── 2. Shard ID coverage ──────────────────────────────────────────────────
_n_with_shard    = sum(1 for m in _manifest if m.get("shard_id") is not None)
_n_without_shard = len(_manifest) - _n_with_shard
_coverage        = _n_with_shard / len(_manifest)

print(f"  Shard coverage   : {_n_with_shard:,} / {len(_manifest):,} "
      f"({100*_coverage:.1f}%) samples have shard_id")

if _coverage < MIN_SHARD_COVERAGE:
    # Below 50% → extraction almost certainly never ran or was badly interrupted
    raise RuntimeError(
        f"Only {_n_with_shard:,}/{len(_manifest):,} samples ({100*_coverage:.1f}%) "
        f"have shard_id — below the {100*MIN_SHARD_COVERAGE:.0f}% minimum.\n\n"
        "This indicates Phase 1 §11 (hidden-state extraction) did not run or\n"
        "was interrupted very early. Re-run Phase 1 §11.3 to extract hidden states.\n"
        "It is restart-safe: already-cached samples will be skipped automatically."
    )
elif _n_without_shard > 0:
    # Above 50% but some missing → Phase 1 tokenization failures (normal)
    print(
        f"  INFO: {_n_without_shard:,} samples have no shard_id "
        f"({100*_n_without_shard/len(_manifest):.1f}%).\n"
        f"        These were legitimately skipped during Phase 1 §9.4 tokenization\n"
        f"        (e.g. exceeded max_seq_len, corrupt image, empty response).\n"
        f"        Phase2Dataset will filter them out automatically.\n"
        f"        Effective Phase 2 training set: {_n_with_shard:,} samples."
    )
else:
    print(f"  ok  all {_n_with_shard:,} samples have shard_id")

assert _n_with_shard > 0, (
    "Zero samples have shard_id — Phase 1 §11 must run before Phase 2."
)

# ── 3. hidden_states_meta.json — template hash ────────────────────────────
assert CFG.hidden_states_meta_path.exists(), (
    f"hidden_states_meta.json not found: {CFG.hidden_states_meta_path}\n"
    "Run Phase 1 §11.2 (cache_vlm_hidden_states) to generate this file."
)
_meta = load_json(CFG.hidden_states_meta_path)
_cached_hash = _meta.get("phase1_template_hash", "<missing>")
if _cached_hash != PHASE1_TEMPLATE_HASH:
    raise RuntimeError(
        f"Prompt template hash mismatch!\n"
        f"  Cached (Phase 1) : {_cached_hash!r}\n"
        f"  Current (Phase 2): {PHASE1_TEMPLATE_HASH!r}\n"
        "The §8.1 prompt template in this notebook does not match the one used\n"
        "during Phase 1 hidden-state extraction. Either:\n"
        "  (a) Restore §8.1 to the exact Phase 1 template, or\n"
        "  (b) Re-run Phase 1 §11 with the updated template to rebuild shards."
    )
print(f"  ok  template_hash match: {PHASE1_TEMPLATE_HASH!r}")
print(f"  ok  metadata: {_meta}")

# ── 4. Shard files exist ──────────────────────────────────────────────────
_shard_files = sorted(CFG.hidden_states_dir.glob("shard_*.pt"))
assert len(_shard_files) > 0, (
    f"No shard_*.pt files in {CFG.hidden_states_dir}.\n"
    "Run Phase 1 §11.3 to extract and cache VLM hidden states."
)
print(f"  ok  {len(_shard_files)} shard files in {CFG.hidden_states_dir.name}/")

# ── 5. Shard round-trip shape check ──────────────────────────────────────
_cached_samples = [m for m in _manifest if m.get("shard_id") is not None]
_probe = random.choice(_cached_samples)
_probe_shard_path = CFG.hidden_states_dir / f"{_probe['shard_id']}.pt"
assert _probe_shard_path.exists(), (
    f"Shard file referenced in manifest not found: {_probe_shard_path}\n"
    "The manifest may have been written with a different Drive path."
)
_shard_data = torch.load(_probe_shard_path, map_location="cpu")
assert "hidden_states" in _shard_data and "sample_ids" in _shard_data, (
    f"Shard {_probe_shard_path.name} missing keys: {list(_shard_data.keys())}"
)
_hs = _shard_data["hidden_states"]
assert _hs.ndim == 2 and _hs.shape[1] == CFG.vlm_hidden_dim, (
    f"Unexpected shard shape: {_hs.shape} (expected (N, {CFG.vlm_hidden_dim}))"
)
_row = _probe["row_index"]
assert 0 <= _row < _hs.shape[0], (
    f"row_index {_row} out of range for shard shape {_hs.shape}"
)
_probe_vec = _hs[_row]
assert not _probe_vec.isnan().any(), f"NaN in hidden state for {_probe['sample_id']!r}"
print(f"  ok  shard round-trip: sample={_probe['sample_id']!r}, "
      f"shard={_probe['shard_id']!r}, row={_row}, shape={tuple(_probe_vec.shape)}")

# ── 6. Spot-check source image + segmentation JSON ────────────────────────
_probe2 = random.choice(_cached_samples)
_src_path = CFG.data_dir / _probe2["source_image"]
assert _src_path.exists() and _src_path.stat().st_size > 0, (
    f"Source image missing or empty: {_src_path}\n"
    "Run Phase 0 §7.1 to download images."
)
_tgt_path = CFG.data_dir / _probe2["target_image"]
assert _tgt_path.exists() and _tgt_path.stat().st_size > 0, (
    f"Target image missing or empty: {_tgt_path}"
)
_seg_path = CFG.data_dir / _probe2["segmentation"]
assert _seg_path.exists(), f"Segmentation JSON missing: {_seg_path}"
_seg_data = load_json(_seg_path)
_anns     = _seg_data.get("annotations", [])
print(f"  ok  spot-check sample={_probe2['sample_id']!r}: "
      f"src={_src_path.name}, tgt={_tgt_path.name}, anns={len(_anns)}")

print(f"\n✓ §13.1 passed. Effective training set: {_n_with_shard:,} samples.")
print(f"  ({_n_without_shard:,} samples skipped — Phase 1 tokenization failures, expected)")
_PREFLIGHT_PASSED = True


§13.1  Phase 1 artifact validation
  cache version    : 'v2'
  manifest path    : /content/drive/MyDrive/img_edit_pipeline/data/imgedit_subset/samples_filtered_v2.json
  hidden-states dir: /content/drive/MyDrive/img_edit_pipeline/data/imgedit_subset/vlm_hidden_states_v2
  ckpt dir         : /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16
  ok  manifest: 8,691 samples from samples_filtered_v2.json
  Shard coverage   : 7,818 / 8,691 (90.0%) samples have shard_id
  INFO: 873 samples have no shard_id (10.0%).
        These were legitimately skipped during Phase 1 §9.4 tokenization
        (e.g. exceeded max_seq_len, corrupt image, empty response).
        Phase2Dataset will filter them out automatically.
        Effective Phase 2 training set: 7,818 samples.
  ok  template_hash match: 'e8964e0ced3f9891'
  ok  metadata: {'phase1_template_hash': 'e8964e0ced3f9891', 'vlm_model_id': 'Qwen/Qwen2.5-VL-3B-Instruct', 'hidden_dim': 2048, 'total_samples': 7818, 'n_shards

## §14 — Phase 2: Model Loading

### §14.1 — Load SD Inpainting Pipeline Components
Loads VAE, UNet (9-channel), CLIP text encoder, and DDPMScheduler. VAE and CLIP are immediately frozen. UNet LoRA is attached in §14.4.

In [ ]:
# ── §14.1  Load Stable Diffusion inpainting pipeline components ──────────
#
# We load the four components separately (not as a complete pipeline) because:
#   - VAE and CLIP are frozen; UNet gets LoRA adapters
#   - We build a custom training loop (not the diffusers inference pipeline)
#   - We need direct access to each component's forward method
#
# Components loaded:
#   vae        — AutoencoderKL: encodes/decodes images to/from latent space
#   unet       — UNet2DConditionModel: 9-channel inpainting variant (in_channels=9)
#   clip       — CLIPTextModel: text encoder for conditioning
#   tokenizer  — CLIPTokenizer: tokenizer for CLIP
#   scheduler  — DDPMScheduler: training-time noise scheduler
#
# SD 1.5 inpainting (runwayml/stable-diffusion-inpainting):
#   UNet in_channels = 9 (not 4): extra 5 channels carry mask + masked-image latent
#   VAE uses latent scaling factor 0.18215 (stored in CFG.vae_scale_factor)
#
# FROZEN (no gradient computation):
#   VAE: image ↔ latent conversion; no task-specific learning needed
#   CLIP: instruction encoding; SD 1.5 text encoder; frozen by convention
#
# TRAINABLE:
#   UNet LoRA adapters: adapt cross-attention to our combined conditioning
#   VLMProjectionAdapter: maps VLM → CLIP embedding space (defined in §14.3)

import torch
from diffusers import (
    AutoencoderKL,
    UNet2DConditionModel,
    DDPMScheduler,
)
from transformers import CLIPTextModel, CLIPTokenizer

assert "_PREFLIGHT_PASSED" in dir() and _PREFLIGHT_PASSED, (
    "§13.1 preflight check has not passed. Run §13.1 before loading models."
)

print("Loading SD inpainting components from:", CFG.sd_model_id)
print("(First run downloads ~5 GB to HF cache; subsequent runs are instant)")

# ── VAE ──────────────────────────────────────────────────────────────────
vae = AutoencoderKL.from_pretrained(
    CFG.sd_model_id,
    subfolder="vae",
    torch_dtype=torch.float32,   # VAE encode in fp32 to avoid precision loss
)
vae.eval()
vae.requires_grad_(False)
vae = vae.to(DEVICE)
print(f"  ok  VAE loaded: {type(vae).__name__}")

# ── UNet (9-channel inpainting) ───────────────────────────────────────────
unet = UNet2DConditionModel.from_pretrained(
    CFG.sd_model_id,
    subfolder="unet",
    torch_dtype=torch.float32,   # load fp32; mixed-precision cast happens in training
)
# Verify this is the 9-channel inpainting variant — NEVER assume
assert unet.config.in_channels == 9, (
    f"Expected UNet in_channels=9 (inpainting variant), got {unet.config.in_channels}.\n"
    f"Model ID: {CFG.sd_model_id}\n"
    "Use runwayml/stable-diffusion-inpainting, not the base SD 1.5 model."
)
unet.train()   # will be put in eval for non-LoRA params after LoRA attachment
unet = unet.to(DEVICE)
print(f"  ok  UNet loaded: in_channels={unet.config.in_channels}, "
      f"cross_attention_dim={unet.config.cross_attention_dim}")
assert unet.config.cross_attention_dim == CFG.sd_cross_attn_dim, (
    f"UNet cross_attention_dim={unet.config.cross_attention_dim} "
    f"!= CFG.sd_cross_attn_dim={CFG.sd_cross_attn_dim}"
)

# ── CLIP text encoder + tokenizer ────────────────────────────────────────
tokenizer = CLIPTokenizer.from_pretrained(CFG.sd_model_id, subfolder="tokenizer")
clip = CLIPTextModel.from_pretrained(
    CFG.sd_model_id,
    subfolder="text_encoder",
    torch_dtype=torch.float32,
)
clip.eval()
clip.requires_grad_(False)
clip = clip.to(DEVICE)
print(f"  ok  CLIP text encoder loaded: hidden_size={clip.config.hidden_size}")
assert clip.config.hidden_size == CFG.sd_cross_attn_dim, (
    f"CLIP hidden_size={clip.config.hidden_size} != CFG.sd_cross_attn_dim={CFG.sd_cross_attn_dim}"
)

# ── DDPMScheduler — training noise scheduler ─────────────────────────────
# Load from the model's scheduler config to ensure consistent beta schedule.
# DDPM is required for training (gives add_noise() method).
# The inference pipeline uses PNDM/DDIM — different scheduler, same trained UNet.
noise_scheduler = DDPMScheduler.from_pretrained(
    CFG.sd_model_id,
    subfolder="scheduler",
)
# Override num_train_timesteps from CFG for consistency
noise_scheduler.config.num_train_timesteps = CFG.phase2_num_train_timesteps
print(f"  ok  DDPMScheduler: num_train_timesteps={noise_scheduler.config.num_train_timesteps}, "
      f"beta_schedule={noise_scheduler.config.beta_schedule}")

print("\n✓ All SD components loaded.")
print(f"  VAE:       frozen,   device={next(vae.parameters()).device}")
print(f"  UNet:      trainable (LoRA in §14.4),  device={next(unet.parameters()).device}")
print(f"  CLIP:      frozen,   device={next(clip.parameters()).device}")
print(f"  Scheduler: DDPMScheduler (training)")


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading SD inpainting components from: runwayml/stable-diffusion-inpainting
(First run downloads ~5 GB to HF cache; subsequent runs are instant)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

An error occurred while trying to fetch runwayml/stable-diffusion-inpainting: runwayml/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: runwayml/stable-diffusion-inpainting
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ok  CLIP text encoder loaded: hidden_size=768


scheduler_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

  ok  DDPMScheduler: num_train_timesteps=1000, beta_schedule=scaled_linear

✓ All SD components loaded.
  VAE:       frozen,   device=cuda:0
  UNet:      trainable (LoRA in §14.4),  device=cuda:0
  CLIP:      frozen,   device=cuda:0
  Scheduler: DDPMScheduler (training)


### §14.2 — Verify 9-Channel UNet Input + Attention Module Names
Prints all `to_q/k/v/to_out.0` module names and runs a mock forward pass.

In [ ]:
# ── §14.2  Verify 9-channel UNet input + print attention module names ─────
#
# SPEC REQUIREMENT: "Do not assume model module names; print and verify them."
#
# This cell:
#   1. Asserts UNet in_channels == 9
#   2. Lists all attention Linear modules — used to confirm target_modules for LoRA
#   3. Runs a mock forward pass to confirm the 9-channel input works end-to-end
#      (before LoRA attachment which changes module structure)

import torch

print("UNet config:")
print(f"  in_channels            = {unet.config.in_channels}")   # must be 9
print(f"  cross_attention_dim    = {unet.config.cross_attention_dim}")
print(f"  sample_size (latent)   = {unet.config.sample_size}")   # 64 = 512/8

# ── Print attention Linear module names ──────────────────────────────────
print("\nAttention Linear modules (first 20 — verify to_q/k/v/out.0 names):")
_attn_modules = []
for name, mod in unet.named_modules():
    if isinstance(mod, torch.nn.Linear):
        for suffix in ("to_q", "to_k", "to_v", "to_out.0"):
            if name.endswith(suffix):
                _attn_modules.append((name, type(mod).__name__, mod.in_features, mod.out_features))
                break
for nm, tp, inf, outf in _attn_modules[:20]:
    print(f"  {nm}  [{inf} → {outf}]")
if len(_attn_modules) > 20:
    print(f"  ... ({len(_attn_modules)} total attention Linear modules)")
assert len(_attn_modules) > 0, (
    "No attention Linear modules found matching to_q/k/v/to_out.0 suffixes.\n"
    "CFG.sd_lora_target_modules may need updating: "
    f"current value = {CFG.sd_lora_target_modules}"
)
print(f"\n  Total attention modules matching target_modules: {len(_attn_modules)}")
print(f"  target_modules = {CFG.sd_lora_target_modules}")

# ── Mock forward pass — verify 9-channel input ───────────────────────────
print("\nRunning mock forward pass (9-channel UNet)...")
_B, _H, _W = 1, 64, 64   # batch=1, latent 512/8=64
_mock_latent  = torch.randn(_B, 4, _H, _W, device=DEVICE)    # noisy target latent
_mock_mask    = torch.zeros(_B, 1, _H, _W, device=DEVICE)     # mask at latent size
_mock_src_lat = torch.randn(_B, 4, _H, _W, device=DEVICE)    # masked-src latent
_mock_input   = torch.cat([_mock_latent, _mock_mask, _mock_src_lat], dim=1)  # (1,9,64,64)

assert _mock_input.shape == (_B, 9, _H, _W), (
    f"9-channel input shape wrong: {_mock_input.shape}"
)

_mock_cond    = torch.randn(_B, 78, CFG.sd_cross_attn_dim, device=DEVICE)   # (1,78,768)
_mock_t       = torch.tensor([500], device=DEVICE)

with torch.no_grad():
    _mock_out = unet(_mock_input, _mock_t, encoder_hidden_states=_mock_cond)
_mock_pred = _mock_out.sample
assert _mock_pred.shape == (_B, 4, _H, _W), (
    f"UNet output shape {_mock_pred.shape} != ({_B}, 4, {_H}, {_W})"
)
print(f"  ok  input shape:  {tuple(_mock_input.shape)}")
print(f"  ok  cond shape:   {tuple(_mock_cond.shape)}")
print(f"  ok  output shape: {tuple(_mock_pred.shape)}")
print("\n✓ §14.2 passed — 9-channel UNet verified, attention module names confirmed.")


UNet config:
  in_channels            = 9
  cross_attention_dim    = 768
  sample_size (latent)   = 64

Attention Linear modules (first 20 — verify to_q/k/v/out.0 names):
  down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q  [320 → 320]
  down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k  [320 → 320]
  down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_v  [320 → 320]
  down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0  [320 → 320]
  down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_q  [320 → 320]
  down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_k  [768 → 320]
  down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_v  [768 → 320]
  down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_out.0  [320 → 320]
  down_blocks.0.attentions.1.transformer_blocks.0.attn1.to_q  [320 → 320]
  down_blocks.0.attentions.1.transformer_blocks.0.attn1.to_k  [320 → 320]
  down_blocks.0.attentions.1.transformer_blocks.0.attn1.to_v  [320 → 320]
  down_

### §14.3 — VLMProjectionAdapter
2-layer MLP: `(2048,) → (1536,) → (1, 768)`. Concatenated with CLIP (77, 768) → combined conditioning (78, 768).

In [ ]:
# ── §14.3  VLMProjectionAdapter — definition ─────────────────────────────
#
# Projects the VLM mean-pooled hidden state (2048,) to a single
# cross-attention token (1, 768) that is concatenated with the CLIP
# text embeddings (77, 768) to form the combined conditioning (78, 768).
#
# Architecture: Linear → LayerNorm → GELU → Linear
# Why a 2-layer MLP (not just a single Linear)?
#   A single linear projection can only rotate/scale the input subspace.
#   The VLM and CLIP embeddings occupy very different representation spaces
#   (different training objectives, architectures, and vocabularies).
#   One hidden layer with GELU nonlinearity is sufficient to span this gap
#   without overfitting on the small-to-moderate training set.
#
# Why intermediate_dim = sd_cross_attn_dim * 2 (1536)?
#   Larger intermediate gives more expressive power without excessive parameters.
#   With vlm_dim=2048 → intermediate=1536 → sd_dim=768:
#     params = (2048*1536 + 1536) + (1536*768 + 768) ≈ 4.3M — manageable.
#
# Input:  (batch, vlm_hidden_dim=2048)  — mean-pooled last hidden layer
# Output: (batch, 1, sd_cross_attn_dim=768) — single conditioning token

import torch
import torch.nn as nn


class VLMProjectionAdapter(nn.Module):
    """Projects VLM mean-pooled hidden state → SD cross-attention embedding.

    Input:  (batch, vlm_dim)        float32
    Output: (batch, 1, sd_dim)      float32

    Designed to be concatenated with CLIP text embeddings along sequence dim:
        combined = cat([clip_embeds(batch,77,768), vlm_proj(batch,1,768)], dim=1)
        → (batch, 78, 768)  — passed as encoder_hidden_states to UNet
    """

    def __init__(
        self,
        vlm_dim: int = None,
        sd_dim:  int = None,
    ):
        super().__init__()
        vlm_dim = vlm_dim or CFG.vlm_hidden_dim        # 2048
        sd_dim  = sd_dim  or CFG.sd_cross_attn_dim     # 768
        intermediate_dim = sd_dim * 2                   # 1536

        self.proj = nn.Sequential(
            nn.Linear(vlm_dim, intermediate_dim, bias=True),
            nn.LayerNorm(intermediate_dim),
            nn.GELU(),
            nn.Linear(intermediate_dim, sd_dim, bias=True),
        )

        # Weight init: small normal to avoid dominating CLIP embeddings at start
        for layer in self.proj:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, std=0.02)
                nn.init.zeros_(layer.bias)

    def forward(self, vlm_hidden: torch.Tensor) -> torch.Tensor:
        """
        Args:
            vlm_hidden: (batch, vlm_dim) float32 — pre-cached hidden state

        Returns:
            (batch, 1, sd_dim) float32 — single conditioning token
        """
        assert vlm_hidden.ndim == 2, (
            f"VLMProjectionAdapter expects (batch, vlm_dim), got {vlm_hidden.shape}"
        )
        out = self.proj(vlm_hidden)        # (batch, sd_dim)
        return out.unsqueeze(1)            # (batch, 1, sd_dim)


# Instantiate and move to device
vlm_adapter = VLMProjectionAdapter().to(DEVICE)
vlm_adapter.train()   # adapter is fully trainable

# Count parameters
_adapter_params = sum(p.numel() for p in vlm_adapter.parameters() if p.requires_grad)
print(f"VLMProjectionAdapter defined:")
print(f"  vlm_dim={CFG.vlm_hidden_dim} → intermediate={CFG.sd_cross_attn_dim*2} → sd_dim={CFG.sd_cross_attn_dim}")
print(f"  Trainable params: {_adapter_params:,}")

# ── Shape smoke test ──────────────────────────────────────────────────────
_test_input = torch.randn(2, CFG.vlm_hidden_dim, device=DEVICE)
_test_out   = vlm_adapter(_test_input)
assert _test_out.shape == (2, 1, CFG.sd_cross_attn_dim), (
    f"VLMProjectionAdapter output shape {_test_out.shape} != (2, 1, {CFG.sd_cross_attn_dim})"
)
print(f"  ok  shape: ({2}, {CFG.vlm_hidden_dim}) → {tuple(_test_out.shape)}")
print("\n✓ VLMProjectionAdapter instantiated and shape-verified.")


VLMProjectionAdapter defined:
  vlm_dim=2048 → intermediate=1536 → sd_dim=768
  Trainable params: 4,330,752
  ok  shape: (2, 2048) → (2, 1, 768)

✓ VLMProjectionAdapter instantiated and shape-verified.


### §14.4 — Attach UNet LoRA
Attaches PEFT LoRA adapters to UNet attention layers. Verifies non-zero trainable params and that LoRA exists in cross-attention.

In [ ]:
# ── §14.4  Attach UNet LoRA adapters ────────────────────────────────────
#
# Uses PEFT LoraConfig to inject LoRA adapters into the UNet's attention
# Linear modules. Only the LoRA delta weights (rank decomposition) are trained.
# The base UNet weights remain frozen.
#
# Target modules: to_q, to_k, to_v, to_out.0
#   These are the query/key/value/output projections in both:
#   - Self-attention (attn1): learns to attend to image-space features
#   - Cross-attention (attn2): learns to attend to our combined conditioning
#
# Why LoRA on cross-attention too:
#   Cross-attention is where the conditioning (CLIP + VLM) interacts with image
#   features. Fine-tuning cross-attention allows the model to learn the new
#   78-token conditioning structure (77 CLIP + 1 VLM), which is different from
#   the 77-token CLIP-only conditioning the base model was trained on.
#
# WARNING — stale variable check:
#   If you re-run this cell, get_peft_model() would wrap an already-PEFT model,
#   adding double LoRA. Guard against this by checking for PEFT wrapping first.

from peft import LoraConfig, get_peft_model

# ── torchao preflight ────────────────────────────────────────────────────
# PEFT's dispatch_torchao raises ImportError (not a soft failure) if torchao
# is present but < 0.16.0. Check now so the user gets an actionable message
# instead of a cryptic traceback buried inside PEFT's module walk.
import importlib.metadata as _imeta

_tao_installed = None
try:
    _tao_installed = _imeta.version("torchao")
except _imeta.PackageNotFoundError:
    pass  # not installed → PEFT standard dispatcher will be used, no issue

if _tao_installed is not None:
    def _parse_ver(v):
        parts = v.split("+")[0].split(".")
        return tuple(int(x) for x in parts[:3] if x.isdigit())
    _tao_tuple = _parse_ver(_tao_installed)
    if _tao_tuple < (0, 16, 0):
        raise RuntimeError(
            f"\n[§14.4 torchao preflight FAILED]\n"
            f"  torchao=={_tao_installed} is installed but PEFT requires >= 0.16.0.\n"
            f"  PEFT's LoRA dispatcher will raise ImportError at get_peft_model().\n"
            f"\n  Fix: re-run §1.1 (which upgrades torchao>=0.16.0), then:\n"
            f"    Runtime → Restart session, then re-run §1.2 through §14.4."
        )
    print(f"  torchao=={_tao_installed} — satisfies PEFT >= 0.16.0 requirement ✓")
else:
    print("  torchao not installed — PEFT will use standard Linear dispatcher ✓")


# Guard against re-wrapping on cell re-run
if hasattr(unet, "peft_config"):
    print("UNet already has PEFT config — skipping re-wrapping.")
    print("  (Re-run from §14.1 to get a fresh UNet if you need to change LoRA config.)")
else:
    _lora_config = LoraConfig(
        r              = CFG.sd_lora_r,
        lora_alpha     = CFG.sd_lora_alpha,
        lora_dropout   = CFG.sd_lora_dropout,
        target_modules = list(CFG.sd_lora_target_modules),
        bias           = "none",
    )

    unet = get_peft_model(unet, _lora_config)
    print(f"LoRA attached to UNet:")
    print(f"  r={CFG.sd_lora_r}, alpha={CFG.sd_lora_alpha}, dropout={CFG.sd_lora_dropout}")
    print(f"  target_modules={CFG.sd_lora_target_modules}")

# ── Verify trainable parameters ───────────────────────────────────────────
_unet_total     = sum(p.numel() for p in unet.parameters())
_unet_trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f"  UNet total params     : {_unet_total:,}")
print(f"  UNet trainable params : {_unet_trainable:,} ({100*_unet_trainable/_unet_total:.2f}%)")

assert _unet_trainable > 0, "No trainable UNet parameters after LoRA attachment — check target_modules."
assert _unet_trainable < _unet_total, (
    "ALL UNet params are trainable — LoRA attachment may have failed to freeze base weights."
)

# ── Verify LoRA layers exist in expected locations ────────────────────────
# Check cross-attention specifically (attn2) since that's where VLM conditioning flows
_n_lora_cross = 0
for name, module in unet.named_modules():
    if "attn2" in name and hasattr(module, "lora_A"):
        _n_lora_cross += 1
print(f"  LoRA cross-attention modules (attn2): {_n_lora_cross}")
assert _n_lora_cross > 0, (
    "No LoRA adapters found in cross-attention (attn2) modules.\n"
    "Check that target_modules includes the correct suffix names (to_q, etc.)."
)

print("\n✓ UNet LoRA verified — trainable=LoRA deltas only, base weights frozen.")


  torchao==0.17.0 — satisfies PEFT >= 0.16.0 requirement ✓
LoRA attached to UNet:
  r=16, alpha=32, dropout=0.0
  target_modules=('to_q', 'to_k', 'to_v', 'to_out.0')
  UNet total params     : 862,724,100
  UNet trainable params : 3,188,736 (0.37%)
  LoRA cross-attention modules (attn2): 64

✓ UNet LoRA verified — trainable=LoRA deltas only, base weights frozen.


### §14.5 — Architecture Smoke Test
Full forward chain: VLM hidden state → adapter → CLIP → combined cond (78,768) → VAE encode → 9-ch UNet input → noise prediction → MSE loss. All shapes asserted.

In [ ]:
# ── §14.5  Architecture smoke test (all components together) ───────────────
#
# SPEC REQUIREMENT: "Before writing any training loop, first ensure that the
# notebook can [run] one forward pass through the diffusion UNet with correct shapes."
#
# This cell validates the FULL forward pass chain:
#   VLM hidden state → VLMProjectionAdapter → (1, 768)
#   Instruction text → CLIP text encoder    → (77, 768)
#   Concatenate                              → (78, 768)  [combined conditioning]
#   Source/target images → VAE encode       → latent (4, H/8, W/8)
#   Mask → resize                           → (1, H/8, W/8)
#   Construct 9-ch input                    → (9, H/8, W/8)
#   Add DDPM noise                          → noisy_tgt
#   UNet forward pass                       → noise prediction (4, H/8, W/8)
#   MSE loss                                → scalar

import torch, torch.nn.functional as F
from PIL import Image as _PIL_Image
import numpy as np

_B   = 1
_RES = CFG.phase2_resolution   # 512
_LAT = _RES // 8               # 64

print("=" * 60)
print("§14.5  Architecture smoke test (single sample, no grad)")
print("=" * 60)

# ── Step 1: VLM hidden state (simulates loading from shard) ──────────────
_vlm_hs = torch.randn(_B, CFG.vlm_hidden_dim, device=DEVICE)        # (1, 2048)
print(f"  [1] VLM hidden state    : {tuple(_vlm_hs.shape)}")

# ── Step 2: VLM projection ────────────────────────────────────────────────
_vlm_adapter_was_training = vlm_adapter.training
vlm_adapter.eval()
with torch.no_grad():
    _vlm_proj = vlm_adapter(_vlm_hs)                                  # (1, 1, 768)
vlm_adapter.train(_vlm_adapter_was_training)
assert _vlm_proj.shape == (_B, 1, CFG.sd_cross_attn_dim), (
    f"VLM projection shape {_vlm_proj.shape} != ({_B}, 1, {CFG.sd_cross_attn_dim})"
)
print(f"  [2] VLM projected       : {tuple(_vlm_proj.shape)}")

# ── Step 3: CLIP text encoding ────────────────────────────────────────────
_instructions = ["change the shirt to blue"]
_clip_tokens  = tokenizer(
    _instructions, padding="max_length", truncation=True,
    max_length=77, return_tensors="pt"
).to(DEVICE)
with torch.no_grad():
    _clip_embeds = clip(**_clip_tokens).last_hidden_state              # (1, 77, 768)
assert _clip_embeds.shape == (_B, 77, CFG.sd_cross_attn_dim), (
    f"CLIP embeds shape {_clip_embeds.shape} != ({_B}, 77, {CFG.sd_cross_attn_dim})"
)
print(f"  [3] CLIP text embeds    : {tuple(_clip_embeds.shape)}")

# ── Step 4: Combined conditioning ─────────────────────────────────────────
_combined_cond = torch.cat([_clip_embeds, _vlm_proj], dim=1)          # (1, 78, 768)
assert _combined_cond.shape == (_B, 78, CFG.sd_cross_attn_dim), (
    f"Combined cond shape {_combined_cond.shape} != ({_B}, 78, {CFG.sd_cross_attn_dim})"
)
print(f"  [4] Combined cond       : {tuple(_combined_cond.shape)}")

# ── Step 5: VAE encode source + target (mock images) ─────────────────────
_mock_src_img = torch.rand(_B, 3, _RES, _RES, device=DEVICE) * 2 - 1  # [-1,1]
_mock_tgt_img = torch.rand(_B, 3, _RES, _RES, device=DEVICE) * 2 - 1

with torch.no_grad():
    _tgt_latents = vae.encode(_mock_tgt_img).latent_dist.sample() * CFG.vae_scale_factor
    assert _tgt_latents.shape == (_B, 4, _LAT, _LAT), (
        f"Target latent shape {_tgt_latents.shape} != ({_B}, 4, {_LAT}, {_LAT})"
    )
    print(f"  [5] Target latents      : {tuple(_tgt_latents.shape)}")

# ── Step 6: Mask + masked source ──────────────────────────────────────────
_mock_mask_img = torch.zeros(_B, 1, _RES, _RES, device=DEVICE)
_mock_mask_img[:, :, 100:400, 150:350] = 1.0   # inpaint region
_masked_src    = _mock_src_img * (1.0 - _mock_mask_img)  # zero out inpaint region

with torch.no_grad():
    _masked_src_latents = vae.encode(_masked_src).latent_dist.sample() * CFG.vae_scale_factor
    assert _masked_src_latents.shape == (_B, 4, _LAT, _LAT)

_mask_latent = F.interpolate(_mock_mask_img, size=(_LAT, _LAT), mode="nearest")  # (1,1,64,64)
assert _mask_latent.shape == (_B, 1, _LAT, _LAT)
print(f"  [6] Mask (latent size)  : {tuple(_mask_latent.shape)}")
print(f"      Masked-src latents  : {tuple(_masked_src_latents.shape)}")

# ── Step 7: Add noise + build 9-channel UNet input ────────────────────────
_noise     = torch.randn_like(_tgt_latents)
_timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (_B,), device=DEVICE)
_noisy_tgt = noise_scheduler.add_noise(_tgt_latents, _noise, _timesteps)

_unet_input = torch.cat([_noisy_tgt, _mask_latent, _masked_src_latents], dim=1)  # (1, 9, 64, 64)
assert _unet_input.shape == (_B, 9, _LAT, _LAT), (
    f"UNet input shape {_unet_input.shape} != ({_B}, 9, {_LAT}, {_LAT})"
)
print(f"  [7] UNet 9-ch input     : {tuple(_unet_input.shape)}")
print(f"      noisy_tgt(4) | mask(1) | masked_src(4) = 9 channels confirmed")

# ── Step 8: UNet forward pass ──────────────────────────────────────────────
unet.eval()
with torch.no_grad():
    _noise_pred = unet(_unet_input, _timesteps, encoder_hidden_states=_combined_cond).sample
unet.train()
assert _noise_pred.shape == (_B, 4, _LAT, _LAT), (
    f"Noise prediction shape {_noise_pred.shape} != ({_B}, 4, {_LAT}, {_LAT})"
)
print(f"  [8] Noise prediction    : {tuple(_noise_pred.shape)}")

# ── Step 9: Loss ───────────────────────────────────────────────────────────
_loss = F.mse_loss(_noise_pred.float(), _noise.float())
assert not _loss.isnan(), "Loss is NaN — check model initialization"
assert not _loss.isinf(), "Loss is Inf — check input normalization"
print(f"  [9] MSE loss            : {_loss.item():.4f}")

print("\n✓ §14.5 PASSED — full forward chain verified with correct shapes.")
print("  Proceed to §15 (dataset) → §16 (conditioning) → §18 (training loop).")


§14.5  Architecture smoke test (single sample, no grad)
  [1] VLM hidden state    : (1, 2048)
  [2] VLM projected       : (1, 1, 768)
  [3] CLIP text embeds    : (1, 77, 768)
  [4] Combined cond       : (1, 78, 768)
  [5] Target latents      : (1, 4, 64, 64)
  [6] Mask (latent size)  : (1, 1, 64, 64)
      Masked-src latents  : (1, 4, 64, 64)
  [7] UNet 9-ch input     : (1, 9, 64, 64)
      noisy_tgt(4) | mask(1) | masked_src(4) = 9 channels confirmed
  [8] Noise prediction    : (1, 4, 64, 64)
  [9] MSE loss            : 0.0087

✓ §14.5 PASSED — full forward chain verified with correct shapes.
  Proceed to §15 (dataset) → §16 (conditioning) → §18 (training loop).


## §15 — Phase 2: Dataset

### §15.1 — Phase2Dataset + ShardCache
Loads source/target images, decodes ground-truth RLE mask (NOT SAM2), and retrieves VLM hidden state from shard cache.

In [ ]:
# ── §15.1  Phase2Dataset — loads images, masks, and VLM hidden states ────
#
# Shard loading strategy:
#   ShardCache wraps the hidden-state shard files. Each shard is loaded
#   on first access and kept in memory. With shard_size=5000 and vlm_dim=2048,
#   each shard is ~40 MB (float32) — all shards fit in CPU memory for 10K samples.
#
# Mask source (TRAINING ONLY):
#   Ground-truth RLE from segmentation JSON via get_training_mask().
#   Phase 3 inference uses SAM2-generated masks — these paths NEVER mix.
#
# Image preprocessing:
#   Both source and target are resized to CFG.phase2_resolution × CFG.phase2_resolution.
#   Pixel values normalized to [-1, 1] for VAE encoding.
#   Mask: 1.0 = inpaint region (edit here), 0.0 = keep region.
#
# Samples without shard_id are skipped (extraction failed in Phase 1).
# Samples with missing source/target images are skipped with a warning.

import torch
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset
from pathlib import Path
from typing import Optional


class ShardCache:
    """Lazy-loaded, in-memory cache of VLM hidden-state shard files.

    Each shard_*.pt file is loaded on first access and kept in a dict.
    Thread-unsafe — intended for single-worker use in training DataLoader
    (use num_workers=0 or ensure each worker has its own ShardCache via
    worker_init_fn when using num_workers>0).
    """

    def __init__(self, hidden_states_dir: Path):
        self.hidden_states_dir = Path(hidden_states_dir)
        self._cache: dict = {}    # shard_id str → {"hidden_states": Tensor, "sample_ids": list}

    def get(self, shard_id: str, row_index: int) -> torch.Tensor:
        """Return the hidden state tensor for one sample.

        Args:
            shard_id:  e.g. "shard_0003"
            row_index: row within the shard (from manifest["row_index"])

        Returns:
            Tensor of shape (vlm_hidden_dim,) dtype=float32 on CPU.

        Raises:
            FileNotFoundError: if shard file is missing.
            IndexError:        if row_index is out of range.
            RuntimeError:      if shard has unexpected shape or contains NaN.
        """
        if shard_id not in self._cache:
            shard_path = self.hidden_states_dir / f"{shard_id}.pt"
            if not shard_path.exists():
                raise FileNotFoundError(
                    f"Shard file not found: {shard_path}\n"
                    "Re-run Phase 1 §11.3 if this shard was never written."
                )
            data = torch.load(shard_path, map_location="cpu")
            assert "hidden_states" in data and "sample_ids" in data, (
                f"Shard {shard_path.name} missing keys: {list(data.keys())}"
            )
            hs = data["hidden_states"]
            assert hs.ndim == 2, f"Shard {shard_id}: hidden_states shape {hs.shape} not 2-D"
            if hs.isnan().any():
                raise RuntimeError(f"Shard {shard_id} contains NaN values")
            self._cache[shard_id] = data
            log.debug(f"ShardCache: loaded {shard_path.name}, shape={tuple(hs.shape)}")

        shard = self._cache[shard_id]
        hs    = shard["hidden_states"]
        if row_index >= hs.shape[0]:
            raise IndexError(
                f"row_index={row_index} out of range for shard {shard_id} "
                f"(shape={tuple(hs.shape)})"
            )
        return hs[row_index].float()   # always return float32


class Phase2Dataset(Dataset):
    """Dataset for Phase 2 inpainting bridge training.

    Loads per-sample:
      - source image (PIL → tensor [-1,1])
      - target image (PIL → tensor [-1,1])
      - ground-truth binary mask (RLE → np.uint8 → tensor {0,1})
      - VLM hidden state from shard cache (float32 tensor)
      - instruction text (str, for CLIP encoding in collator)

    Skips samples missing shard_id, missing images, or failing mask decode.
    __len__ reflects the number of usable samples (counted at init time).

    Args:
        manifest:          List of manifest dicts (from samples_filtered.json).
        data_dir:          Root dir for all relative paths in manifest.
        hidden_states_dir: Dir containing shard_*.pt files.
        resolution:        Target image size (square). Default CFG.phase2_resolution.
    """

    def __init__(
        self,
        manifest: list,
        data_dir: Path,
        hidden_states_dir: Path,
        resolution: int = None,
    ):
        self.data_dir    = Path(data_dir)
        self.resolution  = resolution or CFG.phase2_resolution
        self.shard_cache = ShardCache(hidden_states_dir)

        # Filter to samples with shard_id (Phase 1 extraction succeeded)
        _total = len(manifest)
        self.samples = [m for m in manifest if m.get("shard_id") is not None]
        _skipped = _total - len(self.samples)
        if _skipped > 0:
            log.warning(
                f"Phase2Dataset: {_skipped}/{_total} samples missing shard_id — skipped. "
                "These had VLM extraction failures in Phase 1 §11."
            )
        log.info(f"Phase2Dataset: {len(self.samples):,} usable samples")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Optional[dict]:
        """Return one training sample or None on failure.

        Callers (Phase2DataCollator) must handle None by skipping the sample.
        Returning None instead of raising prevents a single bad sample from
        crashing the entire training loop.
        """
        entry = self.samples[idx]
        sid   = entry["sample_id"]

        try:
            # ── Load source + target images ──────────────────────────────
            src_path = self.data_dir / entry["source_image"]
            tgt_path = self.data_dir / entry["target_image"]

            if not src_path.exists():
                log.warning(f"[{sid}] Source image missing: {src_path} — skipping")
                return None
            if not tgt_path.exists():
                log.warning(f"[{sid}] Target image missing: {tgt_path} — skipping")
                return None

            src_img = Image.open(src_path).convert("RGB")
            tgt_img = Image.open(tgt_path).convert("RGB")

            # Resize to training resolution (square, BICUBIC)
            src_img = src_img.resize((self.resolution, self.resolution), Image.BICUBIC)
            tgt_img = tgt_img.resize((self.resolution, self.resolution), Image.BICUBIC)

            # PIL → tensor [0,1] → [-1,1]
            src_tensor = TF.to_tensor(src_img) * 2.0 - 1.0   # (3, H, W)
            tgt_tensor = TF.to_tensor(tgt_img) * 2.0 - 1.0   # (3, H, W)

            # ── Load segmentation annotations for RLE mask ───────────────
            seg_path = self.data_dir / entry["segmentation"]
            annotations = []
            seg_load_failed = False
            if seg_path.exists():
                try:
                    seg_data    = load_json(seg_path)
                    annotations = seg_data.get("annotations", [])
                except Exception as e:
                    log.debug(f"[{sid}] Seg JSON load error: {e}")
                    seg_load_failed = True
            else:
                log.debug(f"[{sid}] Seg JSON missing: {seg_path}")
                seg_load_failed = True

            # ── Decode ground-truth training mask ────────────────────────
            # Phase 2 training ONLY — never use SAM2 here.
            mask_np = get_training_mask(
                edit_type   = entry.get("edit_type", "unknown"),
                instruction = entry.get("edit_instruction", ""),
                annotations = annotations,
                target_H    = self.resolution,
                target_W    = self.resolution,
            )
            # mask_np: uint8 (H,W), values 0 or 1
            mask_tensor = torch.from_numpy(mask_np).float().unsqueeze(0)  # (1, H, W)

            # ── Load VLM hidden state from shard ─────────────────────────
            vlm_hidden = self.shard_cache.get(
                entry["shard_id"],
                entry["row_index"],
            )   # (vlm_hidden_dim,) float32 on CPU

            assert vlm_hidden.shape == (CFG.vlm_hidden_dim,), (
                f"[{sid}] Hidden state shape {vlm_hidden.shape} != ({CFG.vlm_hidden_dim},)"
            )

            return {
                "sample_id":   sid,
                "src_image":   src_tensor,      # (3, H, W) float32 [-1,1]
                "tgt_image":   tgt_tensor,      # (3, H, W) float32 [-1,1]
                "mask":        mask_tensor,     # (1, H, W) float32 {0,1}
                "vlm_hidden":  vlm_hidden,      # (2048,) float32 CPU
                "instruction": entry.get("edit_instruction", ""),  # str
            }

        except Exception as e:
            log.warning(f"Phase2Dataset[{idx}] ({sid}) failed: {type(e).__name__}: {e}")
            return None


### §15.2 — Phase2DataCollator
Filters None samples. Returns None if entire batch fails (training loop skips).

In [ ]:
# ── §15.2  Phase2DataCollator — batching with None-sample filtering ───────
#
# Filters out None samples (failed __getitem__ calls) before stacking.
# If ALL samples in a batch fail, returns None — the training loop must
# check for this and skip the step rather than crashing.

from torch.utils.data import DataLoader
from typing import List


def phase2_collate_fn(samples: List[Optional[dict]]) -> Optional[dict]:
    """Collate Phase 2 dataset samples into a batch.

    Filters None samples (failed loads). Returns None if all samples failed.
    The training loop must check: `if batch is None: continue`

    Returns dict with batch-stacked tensors:
        sample_ids  : list[str]                   — kept for logging
        src_images  : (B, 3, H, W) float32        — source images [-1,1]
        tgt_images  : (B, 3, H, W) float32        — target images [-1,1]
        masks       : (B, 1, H, W) float32        — binary mask {0,1}
        vlm_hiddens : (B, vlm_dim) float32         — VLM hidden states
        instructions: list[str]                   — for CLIP encoding in training loop
    """
    valid = [s for s in samples if s is not None]
    if not valid:
        return None

    return {
        "sample_ids":   [s["sample_id"]  for s in valid],
        "src_images":   torch.stack([s["src_image"]  for s in valid]),
        "tgt_images":   torch.stack([s["tgt_image"]  for s in valid]),
        "masks":        torch.stack([s["mask"]       for s in valid]),
        "vlm_hiddens":  torch.stack([s["vlm_hidden"] for s in valid]),
        "instructions": [s["instruction"] for s in valid],
    }


print("Phase2Dataset and phase2_collate_fn defined.")


Phase2Dataset and phase2_collate_fn defined.


### §15.3 — Dataset Smoke Test
Load 5 real samples and assert shapes, value ranges, and mask binary-ness.

In [ ]:
# ── §15.3  Dataset smoke test — 5 samples, full pipeline ─────────────────
#
# Tests: manifest loading, ShardCache, image loading, mask decoding,
# collation, and output tensor shapes — WITHOUT loading the diffusion models.
# Run BEFORE §16 and §17 to catch data problems early.

import torch
from torch.utils.data import DataLoader

print("=" * 60)
print("§15.3  Phase2Dataset smoke test (5 samples)")
print("=" * 60)

_smoke_manifest = [m for m in _manifest if m.get("shard_id") is not None][:10]
assert len(_smoke_manifest) >= 5, (
    f"Need ≥5 manifest samples with shard_id for smoke test, got {len(_smoke_manifest)}"
)

_smoke_ds = Phase2Dataset(
    manifest          = _smoke_manifest,
    data_dir          = CFG.data_dir,
    hidden_states_dir = CFG.hidden_states_dir,
    resolution        = CFG.phase2_resolution,
)
print(f"  Dataset size: {len(_smoke_ds)} usable samples")

# Manual iteration (num_workers=0 avoids multiprocessing issues in Colab)
_smoke_loader = DataLoader(
    _smoke_ds,
    batch_size  = 2,
    shuffle     = False,
    num_workers = 0,
    collate_fn  = phase2_collate_fn,
)

_n_batches  = 0
_n_samples  = 0
for _batch in _smoke_loader:
    if _batch is None:
        print("  WARN: empty batch (all samples failed) — check image paths")
        continue

    _B  = len(_batch["sample_ids"])
    _RES = CFG.phase2_resolution
    _LAT = _RES // 8

    # Shape assertions
    assert _batch["src_images"].shape  == (_B, 3, _RES, _RES), f"src shape wrong: {_batch['src_images'].shape}"
    assert _batch["tgt_images"].shape  == (_B, 3, _RES, _RES), f"tgt shape wrong: {_batch['tgt_images'].shape}"
    assert _batch["masks"].shape       == (_B, 1, _RES, _RES), f"mask shape wrong: {_batch['masks'].shape}"
    assert _batch["vlm_hiddens"].shape == (_B, CFG.vlm_hidden_dim), f"vlm shape wrong: {_batch['vlm_hiddens'].shape}"

    # Value range checks
    assert _batch["src_images"].min() >= -1.1 and _batch["src_images"].max() <= 1.1, (
        f"src_images out of [-1,1]: min={_batch['src_images'].min():.2f}, max={_batch['src_images'].max():.2f}"
    )
    _mask_vals = _batch["masks"].unique().tolist()
    assert all(v in (0.0, 1.0) for v in _mask_vals), f"mask contains non-binary values: {_mask_vals}"

    assert not _batch["vlm_hiddens"].isnan().any(), "NaN in vlm_hiddens"

    print(f"  ok  batch {_n_batches}: samples={_B}, "
          f"src={tuple(_batch['src_images'].shape)}, "
          f"mask_coverage={_batch['masks'].mean():.2%}, "
          f"instructions={[i[:30] for i in _batch['instructions']]}")
    _n_batches += 1
    _n_samples += _B
    if _n_batches >= 2:
        break

assert _n_samples > 0, "Smoke test produced zero valid samples — check image paths and shard files."
print(f"\n✓ §15.3 PASSED — {_n_samples} samples loaded, shapes correct.")
print("  Proceed to §16 (conditioning pipeline).")


§15.3  Phase2Dataset smoke test (5 samples)
  Dataset size: 10 usable samples
  ok  batch 0: samples=2, src=(2, 3, 512, 512), mask_coverage=7.34%, instructions=['Change vest positioned in the ', 'Turn bridge positioned in the ']
  ok  batch 1: samples=2, src=(2, 3, 512, 512), mask_coverage=13.37%, instructions=['Turn cliffs positioned in the ', 'Turn flowers positioned in the']

✓ §15.3 PASSED — 4 samples loaded, shapes correct.
  Proceed to §16 (conditioning pipeline).


## §16 — Conditioning Pipeline
`build_combined_conditioning()` — the single implementation for CLIP+VLM conditioning. Training: `do_cfg=False` → `(B, 78, 768)`. Inference CFG: `do_cfg=True` → `(2B, 78, 768)` with `[uncond || cond]`.

In [ ]:
# ── §16  Conditioning pipeline — CLIP + VLM projection ───────────────────
#
# build_combined_conditioning() is the SINGLE point of conditioning assembly.
# It must be used identically in:
#   - Phase 2 training loop (§18)
#   - Phase 3 inference (import or copy verbatim)
#
# Training mode (do_cfg=False):
#   Returns (batch, 78, 768) — CLIP(77) || VLM_proj(1)
#   No unconditional branch needed; training uses the conditional branch only.
#
# Inference mode (do_cfg=True, called from Phase 3):
#   Returns (2*batch, 78, 768) — [uncond(batch,78,768), cond(batch,78,768)]
#   The UNet doubles the batch for CFG. After UNet forward, the caller splits
#   on the batch dimension and applies: pred = uncond + guidance_scale*(cond-uncond)
#
# SPEC REQUIREMENT: "Guidance behavior must be handled correctly when passing
# prompt_embeds." The unconditional embedding uses:
#   - CLIP("") with max-length padding (standard SD CFG unconditional)
#   - VLM null token: zeros tensor (no VLM conditioning in unconditional branch)
#
# Why concatenate (not add)?
#   SD 1.5 cross-attention expects a sequence of tokens (seq_len, 768).
#   Adding would require both CLIP and VLM to be the same shape.
#   Concatenation adds one new "token" to the attention context, letting the
#   model learn when to attend to VLM vs CLIP information independently.

import torch


def build_combined_conditioning(
    instructions: list,
    vlm_hiddens: torch.Tensor,
    device: str,
    do_cfg: bool = False,
) -> torch.Tensor:
    """Build combined (CLIP + VLM) conditioning tensor for the UNet.

    Args:
        instructions : List[str] of length batch — edit instruction texts.
        vlm_hiddens  : (batch, vlm_hidden_dim) float32 — pre-cached VLM states.
        device       : "cuda" or "cpu".
        do_cfg       : If True, prepend unconditional branch (for inference CFG).
                       MUST be False during training.

    Returns:
        Training   (do_cfg=False): (batch, 78, 768) float32
        Inference  (do_cfg=True) : (2*batch, 78, 768) float32
                                    rows 0..B-1 = unconditional
                                    rows B..2B-1 = conditional

    Shape invariant: last two dims are always (78, sd_cross_attn_dim).
    """
    batch = len(instructions)
    assert vlm_hiddens.shape == (batch, CFG.vlm_hidden_dim), (
        f"vlm_hiddens shape {vlm_hiddens.shape} != ({batch}, {CFG.vlm_hidden_dim})"
    )

    # ── CLIP text encoding ────────────────────────────────────────────────
    clip_tokens = tokenizer(
        instructions,
        padding    = "max_length",
        truncation = True,
        max_length = 77,
        return_tensors = "pt",
    ).to(device)
    with torch.no_grad():
        clip_embeds = clip(**clip_tokens).last_hidden_state  # (batch, 77, 768)

    # ── VLM projection ────────────────────────────────────────────────────
    vlm_proj = vlm_adapter(vlm_hiddens.to(device))          # (batch, 1, 768)

    # ── Conditional branch ────────────────────────────────────────────────
    cond = torch.cat([clip_embeds, vlm_proj], dim=1)         # (batch, 78, 768)

    if not do_cfg:
        return cond

    # ── Unconditional branch (inference CFG only) ─────────────────────────
    # Empty-string CLIP: standard SD uncond token (padding → near-zero embeds)
    uncond_tokens = tokenizer(
        [""] * batch,
        padding    = "max_length",
        truncation = True,
        max_length = 77,
        return_tensors = "pt",
    ).to(device)
    with torch.no_grad():
        uncond_clip = clip(**uncond_tokens).last_hidden_state  # (batch, 77, 768)

    # Null VLM: zeros → no VLM-specific signal in unconditional branch
    null_vlm = torch.zeros(batch, 1, CFG.sd_cross_attn_dim, device=device)

    uncond = torch.cat([uncond_clip, null_vlm], dim=1)       # (batch, 78, 768)

    # Stack: [uncond, cond] along batch dimension
    # UNet receives 2*batch; caller applies CFG after UNet forward
    return torch.cat([uncond, cond], dim=0)                  # (2*batch, 78, 768)


# ── Conditioning smoke test ───────────────────────────────────────────────
print("§16  Conditioning pipeline smoke test")

_test_instr  = ["change the shirt to blue", "remove the car"]
_test_vlm_hs = torch.randn(2, CFG.vlm_hidden_dim, device=DEVICE)

# Training mode
_cond_train = build_combined_conditioning(_test_instr, _test_vlm_hs, DEVICE, do_cfg=False)
assert _cond_train.shape == (2, 78, CFG.sd_cross_attn_dim), (
    f"Training cond shape {_cond_train.shape} != (2, 78, {CFG.sd_cross_attn_dim})"
)
print(f"  ok  training cond:   {tuple(_cond_train.shape)}")

# Inference mode (CFG)
_cond_infer  = build_combined_conditioning(_test_instr, _test_vlm_hs, DEVICE, do_cfg=True)
assert _cond_infer.shape == (4, 78, CFG.sd_cross_attn_dim), (
    f"Inference cond shape {_cond_infer.shape} != (4, 78, {CFG.sd_cross_attn_dim})"
)
_uncond_half = _cond_infer[:2]   # rows 0..1 = unconditional
_cond_half   = _cond_infer[2:]   # rows 2..3 = conditional
# Unconditional VLM token should be zeros
assert _uncond_half[:, -1, :].abs().max().item() < 1e-6, (
    "Unconditional VLM token is not zero — check null_vlm construction in build_combined_conditioning"
)
print(f"  ok  inference cond:  {tuple(_cond_infer.shape)} (CFG: uncond||cond)")
print(f"  ok  uncond VLM token is zeros: {_uncond_half[:,-1,:].abs().max().item():.2e}")
print("\n✓ §16 PASSED — combined conditioning pipeline verified.")
print("  Training: (batch, 78, 768) | Inference-CFG: (2*batch, 78, 768)")


§16  Conditioning pipeline smoke test
  ok  training cond:   (2, 78, 768)
  ok  inference cond:  (4, 78, 768) (CFG: uncond||cond)
  ok  uncond VLM token is zeros: 0.00e+00

✓ §16 PASSED — combined conditioning pipeline verified.
  Training: (batch, 78, 768) | Inference-CFG: (2*batch, 78, 768)


## §17 — Pre-Training Forward + Backward Pass Validation
Final gate before training. Runs a complete forward+backward pass on a real sample and asserts gradient flow through both VLMProjectionAdapter and UNet LoRA.

In [ ]:
# ── §17  Single forward pass validation (all spec pre-training checks) ────
#
# SPEC REQUIREMENT: "Before writing any training loop, first ensure that the
# notebook can:
#   ✓ install/import dependencies        (§1)
#   ✓ load models                        (§14)
#   ✓ load one sample                    (§15.3)
#   ✓ format one training example        (§15.1)
#   ✓ extract one hidden-state tensor    (cached: loaded in §15.1)
#   ✓ decode one RLE mask                (§15.1 via get_training_mask)
#   ✓ build one combined conditioning    (§16)
#   ✓ run one forward pass through the diffusion UNet with correct shapes"
#
# This cell is the final gate before §18 (training loop).
# It runs a complete single-sample forward + backward pass to verify
# gradient flow through both the VLMProjectionAdapter and UNet LoRA layers.

import torch, torch.nn.functional as F
from torch.utils.data import DataLoader

print("=" * 60)
print("§17  Pre-training forward + backward pass validation")
print("=" * 60)

# ── Load one real sample from the dataset ────────────────────────────────
_val_ds = Phase2Dataset(
    manifest          = [m for m in _manifest if m.get("shard_id") is not None][:5],
    data_dir          = CFG.data_dir,
    hidden_states_dir = CFG.hidden_states_dir,
)
_val_loader = DataLoader(_val_ds, batch_size=1, collate_fn=phase2_collate_fn)
_val_batch  = None
for _b in _val_loader:
    if _b is not None:
        _val_batch = _b
        break
assert _val_batch is not None, "Could not load any real sample for §17 validation."

_B   = 1
_RES = CFG.phase2_resolution
_LAT = _RES // 8
_VD  = CFG.vlm_hidden_dim

print(f"  Sample: {_val_batch['sample_ids'][0]!r}")

# ── 1. Load sample fields ─────────────────────────────────────────────────
src_imgs  = _val_batch["src_images"].to(DEVICE)    # (1, 3, 512, 512)
tgt_imgs  = _val_batch["tgt_images"].to(DEVICE)
masks     = _val_batch["masks"].to(DEVICE)         # (1, 1, 512, 512) {0,1}
vlm_hs    = _val_batch["vlm_hiddens"].to(DEVICE)   # (1, 2048)
instrs    = _val_batch["instructions"]

print(f"  [1] src_images   : {tuple(src_imgs.shape)}")
print(f"      tgt_images   : {tuple(tgt_imgs.shape)}")
print(f"      masks        : {tuple(masks.shape)}, coverage={masks.mean():.2%}")
print(f"      vlm_hiddens  : {tuple(vlm_hs.shape)}")
print(f"      instruction  : {instrs[0][:60]!r}")

# ── 2. VAE encode ─────────────────────────────────────────────────────────
with torch.no_grad():
    tgt_latents      = vae.encode(tgt_imgs).latent_dist.sample() * CFG.vae_scale_factor
    masked_src       = src_imgs * (1.0 - masks)
    masked_src_lats  = vae.encode(masked_src).latent_dist.sample() * CFG.vae_scale_factor

assert tgt_latents.shape     == (_B, 4, _LAT, _LAT), f"tgt_latents: {tgt_latents.shape}"
assert masked_src_lats.shape == (_B, 4, _LAT, _LAT), f"masked_src_lats: {masked_src_lats.shape}"
print(f"  [2] tgt_latents      : {tuple(tgt_latents.shape)}")
print(f"      masked_src_lats  : {tuple(masked_src_lats.shape)}")

# ── 3. Mask at latent size ────────────────────────────────────────────────
mask_latent = F.interpolate(masks, size=(_LAT, _LAT), mode="nearest")  # (1, 1, 64, 64)
assert mask_latent.shape == (_B, 1, _LAT, _LAT)
print(f"  [3] mask_latent      : {tuple(mask_latent.shape)}")

# ── 4. Add noise ──────────────────────────────────────────────────────────
noise     = torch.randn_like(tgt_latents)
timesteps = torch.randint(
    0, noise_scheduler.config.num_train_timesteps, (_B,), device=DEVICE
)
noisy_tgt = noise_scheduler.add_noise(tgt_latents, noise, timesteps)
print(f"  [4] noisy_tgt_latent : {tuple(noisy_tgt.shape)}, t={timesteps.tolist()}")

# ── 5. Build 9-channel UNet input ─────────────────────────────────────────
unet_input = torch.cat([noisy_tgt, mask_latent, masked_src_lats], dim=1)  # (1, 9, 64, 64)
assert unet_input.shape == (_B, 9, _LAT, _LAT), f"unet_input: {unet_input.shape}"
print(f"  [5] unet_input (9ch) : {tuple(unet_input.shape)}")

# ── 6. Build combined conditioning ────────────────────────────────────────
cond_embeds = build_combined_conditioning(instrs, vlm_hs, DEVICE, do_cfg=False)
assert cond_embeds.shape == (_B, 78, CFG.sd_cross_attn_dim), f"cond_embeds: {cond_embeds.shape}"
print(f"  [6] cond_embeds      : {tuple(cond_embeds.shape)}")

# ── 7. UNet forward pass (with grad) ──────────────────────────────────────
unet.train()
vlm_adapter.train()

noise_pred = unet(unet_input, timesteps, encoder_hidden_states=cond_embeds).sample
assert noise_pred.shape == (_B, 4, _LAT, _LAT), f"noise_pred: {noise_pred.shape}"
print(f"  [7] noise_pred       : {tuple(noise_pred.shape)}")

# ── 8. Loss + backward ────────────────────────────────────────────────────
loss = F.mse_loss(noise_pred.float(), noise.float())
assert not loss.isnan(), "Loss is NaN"
assert not loss.isinf(), "Loss is Inf"
loss.backward()
print(f"  [8] loss             : {loss.item():.6f}")

# Verify gradients flowed to VLMProjectionAdapter and UNet LoRA
_adapter_grad = any(
    p.grad is not None and p.grad.abs().sum() > 0
    for p in vlm_adapter.parameters()
)
_lora_grad = any(
    "lora_" in name and p.grad is not None and p.grad.abs().sum() > 0
    for name, p in unet.named_parameters()
)
assert _adapter_grad, "No gradient in VLMProjectionAdapter — backward did not reach it"
assert _lora_grad,    "No gradient in UNet LoRA layers — check LoRA attachment"
print(f"  [9] Gradient check   : adapter={_adapter_grad}, unet_lora={_lora_grad}")

# Zero grads after test
for p in vlm_adapter.parameters():
    if p.grad is not None: p.grad.zero_()
for p in unet.parameters():
    if p.grad is not None: p.grad.zero_()

print("\n✓ §17 PASSED — full pre-training validation complete.")
print("  All spec pre-training checks satisfied. Proceed to §18 (training loop).")


§17  Pre-training forward + backward pass validation
  Sample: '0000001'
  [1] src_images   : (1, 3, 512, 512)
      tgt_images   : (1, 3, 512, 512)
      masks        : (1, 1, 512, 512), coverage=1.02%
      vlm_hiddens  : (1, 2048)
      instruction  : 'Change vest positioned in the right-central area to red plai'
  [2] tgt_latents      : (1, 4, 64, 64)
      masked_src_lats  : (1, 4, 64, 64)
  [3] mask_latent      : (1, 1, 64, 64)
  [4] noisy_tgt_latent : (1, 4, 64, 64), t=[922]
  [5] unet_input (9ch) : (1, 9, 64, 64)
  [6] cond_embeds      : (1, 78, 768)
  [7] noise_pred       : (1, 4, 64, 64)
  [8] loss             : 0.000224
  [9] Gradient check   : adapter=True, unet_lora=True

✓ §17 PASSED — full pre-training validation complete.
  All spec pre-training checks satisfied. Proceed to §18 (training loop).


## §18 — Phase 2 Training Loop

### §18.1 — Training Function Definition
Defines `run_phase2_training()` and `_save_phase2_checkpoint()`. Mixed precision, linear warmup + cosine decay, gradient accumulation, restart safety.

In [ ]:
# ── §18  Phase 2 Training Loop ───────────────────────────────────────────
#
# Trains two components jointly:
#   1. VLMProjectionAdapter  — learns to map VLM hidden states → SD embedding space
#   2. UNet LoRA adapters    — adapt cross/self-attention to new 78-token conditioning
#
# Frozen: VAE, CLIP text encoder, UNet base weights.
#
# Training objective: epsilon-prediction MSE (standard DDPM objective).
#   predict the noise ε added to the noisy target latent at timestep t.
#
# Mixed precision: torch.autocast("cuda", dtype=torch.float16)
#   VAE encode in fp32 (stable), UNet forward in fp16 (fast).
#   GradScaler prevents fp16 underflow in backward pass.
#
# Restart safety:
#   - Checkpoints saved every CFG.phase2_save_steps steps to CFG.ckpt_phase2
#   - Resume from latest checkpoint by loading adapter + unet_lora weights
#   - Step counter written to checkpoint metadata
#
# Spec requirement — training/inference mask path separation:
#   Training uses ground-truth RLE masks (already loaded by Phase2Dataset).
#   Phase 3 inference will use SAM2-generated masks.
#   These paths are separated at the dataset level (Phase2Dataset vs Phase 3 code).
#   No SAM2 code appears anywhere in Phase 2.

import torch, torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from tqdm.auto import tqdm
import math


def run_phase2_training(
    manifest_path:     Path = None,
    resume_from:       Path = None,  # path to a checkpoint dir to resume from
) -> dict:
    """Execute Phase 2 bridge training.

    Args:
        manifest_path: Path to filtered manifest (default CFG.filtered_manifest_path).
        resume_from:   Path to a checkpoint directory for mid-run resumption.
                       If None, training starts from scratch.

    Returns:
        Summary dict: {epochs, total_steps, final_loss, checkpoint_path}.
    """
    if manifest_path is None:
        manifest_path = CFG.filtered_manifest_path

    # ── Load manifest ─────────────────────────────────────────────────────
    manifest = load_json(manifest_path)
    log.info(f"Training manifest: {len(manifest):,} samples")

    # ── Dataset + DataLoader ──────────────────────────────────────────────
    train_ds = Phase2Dataset(
        manifest          = manifest,
        data_dir          = CFG.data_dir,
        hidden_states_dir = CFG.hidden_states_dir,
        resolution        = CFG.phase2_resolution,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size  = CFG.phase2_batch_size,
        shuffle     = True,
        num_workers = CFG.phase2_num_workers,
        collate_fn  = phase2_collate_fn,
        pin_memory  = True,
        drop_last   = True,   # avoids batch-size-1 issues with LayerNorm
        # NOTE: num_workers > 0 requires ShardCache to be per-process.
        # ShardCache is instantiated inside Phase2Dataset.__init__ so each
        # DataLoader worker gets its own shard cache — safe for fork-based workers.
    )

    # ── Optimizer — trains only adapter + LoRA params ─────────────────────
    _trainable_params = [
        {"params": vlm_adapter.parameters(),
         "lr": CFG.phase2_lr, "name": "vlm_adapter"},
        {"params": [p for p in unet.parameters() if p.requires_grad],
         "lr": CFG.phase2_lr, "name": "unet_lora"},
    ]
    optimizer = AdamW(
        _trainable_params,
        betas   = (0.9, 0.999),
        eps     = 1e-8,
        weight_decay = 0.01,
    )

    # ── Learning rate schedule: linear warmup + cosine decay ──────────────
    _total_steps = (len(train_ds) // CFG.phase2_batch_size) * CFG.phase2_epochs
    _warmup      = CFG.phase2_warmup_steps

    def _lr_lambda(step: int) -> float:
        if step < _warmup:
            return float(step) / max(1, _warmup)   # linear warmup
        progress = float(step - _warmup) / max(1, _total_steps - _warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))  # cosine decay

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

    # ── Mixed precision scaler ─────────────────────────────────────────────
    scaler = GradScaler(enabled=(CFG.phase2_mixed_precision != "no"))

    # ── Gradient accumulation ─────────────────────────────────────────────
    _accum_steps = CFG.phase2_grad_accum

    # ── Resume from checkpoint ────────────────────────────────────────────
    # v1.6: full-state resume. Restores optimizer, scheduler, scaler, RNG
    # so the LR cosine curve and Adam moments continue cleanly. Also
    # validates that the checkpoint was trained against the same VLM
    # cache version that CFG currently points at — a checkpoint trained
    # on v1 hidden states cannot be resumed against v2.
    _global_step = 0
    _start_epoch = 0
    if resume_from is not None:
        resume_from = Path(resume_from)
        if not resume_from.exists():
            raise FileNotFoundError(f"resume_from does not exist: {resume_from}")
        _ckpt_meta = load_json(resume_from / "training_state.json")

        # Cache-version compatibility gate (v1.6)
        _ckpt_cache_v = _ckpt_meta.get("vlm_cache_version", "v1")
        if _ckpt_cache_v != CFG.vlm_cache_version:
            raise RuntimeError(
                f"Cache version mismatch on resume:\n"
                f"  checkpoint was trained on vlm_cache_version={_ckpt_cache_v!r}\n"
                f"  CFG.vlm_cache_version is now {CFG.vlm_cache_version!r}\n"
                "Resuming would silently mix two VLM hidden-state distributions.\n"
                "Either start fresh (do not pass resume_from), or set\n"
                f"CFG.vlm_cache_version back to {_ckpt_cache_v!r}."
            )

        _global_step  = _ckpt_meta.get("global_step", 0)
        _start_epoch  = _ckpt_meta.get("epoch", 0)
        _adapter_path = resume_from / "vlm_adapter.pt"
        _unet_path    = resume_from / "unet_lora.pt"
        if _adapter_path.exists():
            vlm_adapter.load_state_dict(torch.load(_adapter_path, map_location=DEVICE))
            log.info(f"Resumed VLMProjectionAdapter from {_adapter_path}")
        if _unet_path.exists():
            unet.load_state_dict(torch.load(_unet_path, map_location=DEVICE), strict=False)
            log.info(f"Resumed UNet LoRA from {_unet_path}")

        # Optimizer / scheduler / scaler / RNG
        _opt_path = resume_from / "optimizer.pt"
        if _opt_path.exists():
            optimizer.load_state_dict(torch.load(_opt_path, map_location=DEVICE))
            log.info(f"Resumed optimizer state from {_opt_path}")
        _sched_path = resume_from / "scheduler.pt"
        if _sched_path.exists():
            scheduler.load_state_dict(torch.load(_sched_path, map_location="cpu"))
            log.info(f"Resumed LR scheduler state from {_sched_path}")
        else:
            # Fast-forward LambdaLR by global_step so cosine continues correctly.
            for _ in range(_global_step):
                scheduler.step()
            log.warning(
                "scheduler.pt not found — fast-forwarded LambdaLR by "
                f"{_global_step} steps so the cosine schedule resumes "
                "from the right point."
            )
        _scaler_path = resume_from / "scaler.pt"
        if _scaler_path.exists():
            scaler.load_state_dict(torch.load(_scaler_path, map_location="cpu"))
            log.info(f"Resumed GradScaler state from {_scaler_path}")
        _rng_path = resume_from / "rng_state.pt"
        if _rng_path.exists():
            _rng = torch.load(_rng_path, map_location="cpu")
            torch.set_rng_state(_rng["cpu"])
            if torch.cuda.is_available() and "cuda" in _rng:
                torch.cuda.set_rng_state_all(_rng["cuda"])
            log.info(f"Resumed RNG state from {_rng_path}")

        log.info(f"Resuming from epoch={_start_epoch}, step={_global_step}")

    # ── Training loop ─────────────────────────────────────────────────────
    unet.train()
    vlm_adapter.train()
    vae.eval()
    clip.eval()

    _RES = CFG.phase2_resolution
    _LAT = _RES // 8
    _amp_dtype = torch.float16 if CFG.phase2_mixed_precision == "fp16" else torch.bfloat16

    _running_loss  = 0.0
    _steps_in_accum = 0
    _last_logged_loss = 0.0

    optimizer.zero_grad()

    for epoch in range(_start_epoch, CFG.phase2_epochs):
        log.info(f"Epoch {epoch+1}/{CFG.phase2_epochs} — {len(train_loader)} batches")
        _epoch_loss = 0.0
        _epoch_steps = 0

        _pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", unit="batch", leave=True)

        for _batch in _pbar:
            if _batch is None:
                log.debug("Skipping empty batch (all samples failed)")
                continue

            src_imgs  = _batch["src_images"].to(DEVICE)
            tgt_imgs  = _batch["tgt_images"].to(DEVICE)
            masks     = _batch["masks"].to(DEVICE)
            vlm_hs    = _batch["vlm_hiddens"].to(DEVICE)
            instrs    = _batch["instructions"]
            _B        = src_imgs.shape[0]

            try:
                with autocast(dtype=_amp_dtype):
                    # ── VAE encode (fp32 inside autocast for stability) ───
                    with torch.no_grad():
                        tgt_latents     = vae.encode(tgt_imgs.float()).latent_dist.sample()
                        tgt_latents     = tgt_latents * CFG.vae_scale_factor
                        masked_src      = src_imgs.float() * (1.0 - masks.float())
                        masked_src_lats = vae.encode(masked_src).latent_dist.sample()
                        masked_src_lats = masked_src_lats * CFG.vae_scale_factor

                    # ── Noise + timestep ──────────────────────────────────
                    noise     = torch.randn_like(tgt_latents)
                    timesteps = torch.randint(
                        0, noise_scheduler.config.num_train_timesteps,
                        (_B,), device=DEVICE
                    )
                    noisy_tgt = noise_scheduler.add_noise(tgt_latents, noise, timesteps)

                    # ── Mask at latent size ───────────────────────────────
                    mask_latent = F.interpolate(
                        masks.float(), size=(_LAT, _LAT), mode="nearest"
                    )

                    # ── 9-channel UNet input ──────────────────────────────
                    unet_input = torch.cat(
                        [noisy_tgt, mask_latent, masked_src_lats], dim=1
                    )   # (B, 9, 64, 64)

                    # Assert shape every first step per epoch to catch regressions
                    if _epoch_steps == 0:
                        assert unet_input.shape == (_B, 9, _LAT, _LAT), (
                            f"unet_input shape {unet_input.shape} != ({_B},9,{_LAT},{_LAT})"
                        )

                    # ── Combined conditioning (training: do_cfg=False) ────
                    # NOTE: do_cfg=False is mandatory for training.
                    # Passing do_cfg=True would double the batch, causing a
                    # mismatch between unet_input and cond_embeds batch sizes.
                    cond_embeds = build_combined_conditioning(
                        instrs, vlm_hs, DEVICE, do_cfg=False
                    )   # (B, 78, 768)

                    # ── UNet forward → noise prediction ───────────────────
                    noise_pred = unet(
                        unet_input, timesteps,
                        encoder_hidden_states=cond_embeds,
                    ).sample   # (B, 4, 64, 64)

                    # ── Loss: MSE over epsilon (standard DDPM) ────────────
                    loss = F.mse_loss(noise_pred.float(), noise.float())

                # ── Backward + grad accum ──────────────────────────────────
                scaler.scale(loss / _accum_steps).backward()
                _steps_in_accum += 1

                if _steps_in_accum == _accum_steps:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        list(vlm_adapter.parameters()) +
                        [p for p in unet.parameters() if p.requires_grad],
                        CFG.phase2_max_grad_norm,
                    )
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()
                    _steps_in_accum = 0
                    _global_step += 1

                    _running_loss += loss.item()
                    _epoch_loss   += loss.item()
                    _epoch_steps  += 1
                    _last_logged_loss = loss.item()

                    _pbar.set_postfix({
                        "loss": f"{loss.item():.4f}",
                        "lr":   f"{scheduler.get_last_lr()[0]:.2e}",
                        "step": _global_step,
                    })

                    # ── Periodic checkpoint ───────────────────────────────
                    if _global_step > 0 and _global_step % CFG.phase2_save_steps == 0:
                        _ckpt_dir = CFG.ckpt_phase2 / f"step_{_global_step:06d}"
                        _save_phase2_checkpoint(
                            _ckpt_dir, epoch, _global_step, loss.item(),
                            optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                        )

            except Exception as _train_exc:
                log.error(
                    f"Training step failed at epoch={epoch+1}, step={_global_step}: "
                    f"{type(_train_exc).__name__}: {_train_exc}"
                )
                optimizer.zero_grad()
                _steps_in_accum = 0
                continue

        _avg_epoch_loss = _epoch_loss / max(1, _epoch_steps)
        log.info(f"Epoch {epoch+1} done — avg_loss={_avg_epoch_loss:.4f}, steps={_epoch_steps}")

    # ── Final checkpoint ──────────────────────────────────────────────────
    _final_ckpt = CFG.ckpt_phase2 / "final"
    _save_phase2_checkpoint(
        _final_ckpt, CFG.phase2_epochs - 1, _global_step, _last_logged_loss,
        optimizer=optimizer, scheduler=scheduler, scaler=scaler,
    )

    return {
        "epochs":           CFG.phase2_epochs,
        "total_steps":      _global_step,
        "final_loss":       _last_logged_loss,
        "checkpoint_path":  str(_final_ckpt),
    }


def _save_phase2_checkpoint(
    ckpt_dir: Path,
    epoch: int,
    global_step: int,
    loss: float,
    optimizer = None,
    scheduler = None,
    scaler    = None,
) -> None:
    """Save VLMProjectionAdapter and UNet LoRA weights + full training state.

    Does NOT save frozen weights (VAE, CLIP, UNet base) — only the
    delta weights that were trained, plus optimizer/scheduler/scaler/RNG
    so a future `resume_from=...` continues the LR cosine and Adam moments
    correctly instead of restarting them.

    Saves:
        vlm_adapter.pt       — VLMProjectionAdapter state dict
        unet_lora_hf/        — UNet PEFT adapter (HF format) — preferred
        unet_lora.pt         — fallback: LoRA state dict only
        optimizer.pt         — AdamW state (v1.6)
        scheduler.pt         — LambdaLR state (v1.6)
        scaler.pt            — GradScaler state (v1.6)
        rng_state.pt         — torch CPU + CUDA RNG state (v1.6)
        training_state.json  — metadata (epoch, step, loss, config hash, cache version)
    """
    ckpt_dir = Path(ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    torch.save(vlm_adapter.state_dict(), ckpt_dir / "vlm_adapter.pt")

    # Save only LoRA (trainable) parameters from UNet
    # unet.save_pretrained saves PEFT adapter config + weights in HF format
    try:
        unet.save_pretrained(str(ckpt_dir / "unet_lora_hf"))
    except Exception as _e:
        log.warning(f"unet.save_pretrained failed ({_e}) — falling back to torch.save")
        _lora_state = {
            k: v for k, v in unet.state_dict().items()
            if "lora_" in k or "adapter" in k.lower()
        }
        torch.save(_lora_state, ckpt_dir / "unet_lora.pt")
        log.info(f"  Saved {len(_lora_state)} LoRA parameter tensors")

    # v1.6: persist full training state so resume actually resumes.
    if optimizer is not None:
        torch.save(optimizer.state_dict(), ckpt_dir / "optimizer.pt")
    if scheduler is not None:
        torch.save(scheduler.state_dict(), ckpt_dir / "scheduler.pt")
    if scaler is not None:
        torch.save(scaler.state_dict(), ckpt_dir / "scaler.pt")
    _rng = {"cpu": torch.get_rng_state()}
    if torch.cuda.is_available():
        _rng["cuda"] = torch.cuda.get_rng_state_all()
    torch.save(_rng, ckpt_dir / "rng_state.pt")

    save_json({
        "epoch":         epoch,
        "global_step":   global_step,
        "loss":          loss,
        "template_hash": PHASE1_TEMPLATE_HASH,
        "sd_model_id":   CFG.sd_model_id,
        "vlm_hidden_dim": CFG.vlm_hidden_dim,
        "sd_cross_attn_dim": CFG.sd_cross_attn_dim,
        "lora_r":        CFG.sd_lora_r,
        "lora_alpha":    CFG.sd_lora_alpha,
        # v1.6: cache version is part of the resume-compatibility check.
        "vlm_cache_version": CFG.vlm_cache_version,
    }, ckpt_dir / "training_state.json")

    log.info(f"Checkpoint saved: {ckpt_dir}")


print("run_phase2_training() and _save_phase2_checkpoint() defined.")
print("  Trainable: VLMProjectionAdapter + UNet LoRA deltas")
print("  Frozen   : VAE, CLIP, UNet base weights")
print("  Objective: epsilon-prediction MSE (DDPM)")
print("  Saves to : CFG.ckpt_phase2 / step_XXXXXX/ and /final")


run_phase2_training() and _save_phase2_checkpoint() defined.
  Trainable: VLMProjectionAdapter + UNet LoRA deltas
  Frozen   : VAE, CLIP, UNet base weights
  Objective: epsilon-prediction MSE (DDPM)
  Saves to : CFG.ckpt_phase2 / step_XXXXXX/ and /final


### §18.2 — Execute Phase 2 Training

In [ ]:
# ── §18.2  Execute Phase 2 training ──────────────────────────────────────
#
# The run cell is deliberately separated from the function definition so you
# can call it again with different arguments (e.g. resume_from) without
# redefining the function.

import torch
torch.cuda.empty_cache()

# v1.6: when starting fresh against a new VLM hidden-state cache, the
# ckpt_phase2 dir should be empty. If it already has checkpoints we abort
# rather than mixing distributions silently. To resume an in-progress
# v2 run intentionally, set RESUME_FROM below to an explicit step dir.
RESUME_FROM = None    # e.g. CFG.ckpt_phase2 / "step_002000"

print("Starting Phase 2 training...")
print(f"  Cache ver  : {CFG.vlm_cache_version}")
print(f"  Dataset    : {CFG.filtered_manifest_path.name}")
print(f"  Epochs     : {CFG.phase2_epochs}")
print(f"  Batch size : {CFG.phase2_batch_size}  (grad_accum={CFG.phase2_grad_accum})")
print(f"  Effective batch: {CFG.phase2_batch_size * CFG.phase2_grad_accum}")
print(f"  LR         : {CFG.phase2_lr}")
print(f"  Mixed prec : {CFG.phase2_mixed_precision}")
print(f"  Checkpoint : every {CFG.phase2_save_steps} steps → {CFG.ckpt_phase2}")
print(f"  Resume     : {RESUME_FROM}")

if RESUME_FROM is None and CFG.ckpt_phase2.exists() and any(CFG.ckpt_phase2.iterdir()):
    raise RuntimeError(
        f"{CFG.ckpt_phase2} is non-empty and RESUME_FROM is None.\n"
        "Either:\n"
        "  (a) set RESUME_FROM to an explicit step dir to resume that run,\n"
        "  (b) move/delete the existing checkpoints to start fresh, or\n"
        "  (c) bump CFG.vlm_cache_version so a new ckpt dir is used.\n"
        "This guard prevents silently mixing a stale projector + LoRA with\n"
        "newly-extracted VLM hidden states (a train-train mismatch)."
    )

_phase2_summary = run_phase2_training(resume_from=RESUME_FROM)

print("\n" + "=" * 50)
print("Phase 2 training complete.")
for k, v in _phase2_summary.items():
    print(f"  {k:<20s}: {v}")


Starting Phase 2 training...
  Cache ver  : v2
  Dataset    : samples_filtered_v2.json
  Epochs     : 9
  Batch size : 8  (grad_accum=4)
  Effective batch: 32
  LR         : 0.0001
  Mixed prec : fp16
  Checkpoint : every 500 steps → /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16
  Resume     : None


/tmp/ipykernel_8802/62480427.py:103: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(CFG.phase2_mixed_precision != "no"))


Epoch 1:   0%|          | 0/977 [00:00<?, ?batch/s]

/tmp/ipykernel_8802/62480427.py:213: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=_amp_dtype):


Epoch 2:   0%|          | 0/977 [00:00<?, ?batch/s]

...

Epoch 6:   0%|          | 0/977 [00:00<?, ?batch/s]

Epoch 7:   0%|          | 0/977 [00:00<?, ?batch/s]

Epoch 8:   0%|          | 0/977 [00:00<?, ?batch/s]

Epoch 9:   0%|          | 0/977 [00:00<?, ?batch/s]


Phase 2 training complete.
  epochs              : 9
  total_steps         : 2198
  final_loss          : 0.03378298133611679
  checkpoint_path     : /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16/final


## §19 — Phase 2 Post-Training Validation
Loads final checkpoint, runs inference smoke test, and demonstrates correct CFG pattern for Phase 3. Saves output image to `CFG.outputs_eval`.

In [ ]:
# ── §19  Phase 2 Post-Training Validation ────────────────────────────────
#
# Validates the trained checkpoint can be loaded and runs a minimal inference
# smoke test. This is NOT a quantitative benchmark (that's Phase 4).
# It confirms the pipeline is functional before committing the checkpoint.
#
# Inference note on CFG (classifier-free guidance):
#   This section demonstrates the correct CFG pattern for Phase 3.
#   build_combined_conditioning(do_cfg=True) returns (2*B, 78, 768).
#   After UNet forward on the doubled batch, apply:
#     noise_pred = uncond + guidance_scale * (cond - uncond)
#   Do NOT pass do_cfg=True during training — it doubles the batch, causing
#   shape mismatches with the unet_input.

import torch, torch.nn.functional as F
from diffusers import StableDiffusionInpaintPipeline, DDIMScheduler
from PIL import Image as _PILImage
import numpy as np

print("=" * 60)
print("§19  Post-training validation: checkpoint load + inference smoke test")
print("=" * 60)

# ── 1. Load final checkpoint and verify ──────────────────────────────────
_final_ckpt = CFG.ckpt_phase2 / "final"
assert _final_ckpt.exists(), (
    f"Final checkpoint not found: {_final_ckpt}\n"
    "Run §18.2 (training) before §19."
)
_state_meta = load_json(_final_ckpt / "training_state.json")
print(f"  ok  checkpoint found: step={_state_meta['global_step']}, "
      f"loss={_state_meta['loss']:.4f}")

# Verify template hash in checkpoint matches current template
assert _state_meta.get("template_hash") == PHASE1_TEMPLATE_HASH, (
    f"Checkpoint template hash {_state_meta.get('template_hash')!r} "
    f"!= current {PHASE1_TEMPLATE_HASH!r}. "
    "Checkpoint was trained with a different prompt template."
)
print(f"  ok  template hash match: {PHASE1_TEMPLATE_HASH!r}")

# Load adapter weights
_adapter_path = _final_ckpt / "vlm_adapter.pt"
vlm_adapter.load_state_dict(torch.load(_adapter_path, map_location=DEVICE))
vlm_adapter.eval()
print(f"  ok  VLMProjectionAdapter loaded from checkpoint")

# ── 2. Build inference SD pipeline ───────────────────────────────────────
# Use DDIM scheduler for faster inference (fewer steps than DDPM)
print("\nBuilding inference pipeline (DDIM scheduler, 20 steps)...")
_infer_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    CFG.sd_model_id,
    unet           = unet,
    safety_checker = None,
    torch_dtype    = torch.float16,
)
_infer_pipe.scheduler = DDIMScheduler.from_config(_infer_pipe.scheduler.config)
_infer_pipe = _infer_pipe.to(DEVICE)

# ── CRITICAL: diffusers does NOT auto-cast a pre-supplied PeftModel ───────
# When from_pretrained(..., torch_dtype=float16) receives unet=<PeftModel>,
# it skips the dtype conversion because PeftModel is not a UNet2DConditionModel
# instance (confirmed by the warning: "Expected types for unet: (UNet2DConditionModel,),
# got PeftModel"). The rest of the pipeline (VAE, text_encoder) loads fresh in fp16,
# so the UNet stays fp32 while latents are fp16 → RuntimeError at time_embedding.linear_1.
# Fix: explicitly cast UNet to fp16 after the pipeline is assembled.
_infer_pipe.unet = _infer_pipe.unet.to(dtype=torch.float16)
print(f"  ok  UNet cast to fp16 (dtype={next(_infer_pipe.unet.parameters()).dtype})")

_infer_pipe.set_progress_bar_config(disable=True)
print("  ok  inference pipeline ready")

# ── 3. Run smoke test on one real sample ──────────────────────────────────
print("\nRunning inference smoke test on one real sample...")
_val_samples = [m for m in _manifest if m.get("shard_id") is not None][:3]
_smoke_entry = _val_samples[0]

_src_path = CFG.data_dir / _smoke_entry["source_image"]
_seg_path = CFG.data_dir / _smoke_entry["segmentation"]
_src_pil  = _PILImage.open(_src_path).convert("RGB").resize(
    (CFG.phase2_resolution, CFG.phase2_resolution), _PILImage.BICUBIC
)

# Load segmentation + decode mask
_seg_data = load_json(_seg_path)
_anns     = _seg_data.get("annotations", [])
_mask_np  = get_training_mask(
    _smoke_entry.get("edit_type", "unknown"),
    _smoke_entry.get("edit_instruction", ""),
    _anns,
    CFG.phase2_resolution, CFG.phase2_resolution,
)
_mask_pil = _PILImage.fromarray(_mask_np * 255).convert("L")
_instr    = _smoke_entry.get("edit_instruction", "edit the image")

# Build combined conditioning with CFG
# Demonstrates the correct CFG pattern for Phase 3 (do_cfg=True)
_vlm_shard = torch.load(
    CFG.hidden_states_dir / f"{_smoke_entry['shard_id']}.pt",
    map_location="cpu"
)
_vlm_hs_infer = _vlm_shard["hidden_states"][_smoke_entry["row_index"]].unsqueeze(0).to(DEVICE)
# NOTE: The inference pipeline below uses its own CLIP encoding internally
# when passed prompt= (for consistency testing we use it directly).
# A full Phase 3 implementation would intercept the UNet call to inject
# build_combined_conditioning output. This smoke test uses the simplified
# pipe(prompt=...) interface to verify the pipeline runs end-to-end.
_result = _infer_pipe(
    prompt          = _instr,
    image           = _src_pil,
    mask_image      = _mask_pil,
    num_inference_steps = 20,
    guidance_scale  = 7.5,
    height          = CFG.phase2_resolution,
    width           = CFG.phase2_resolution,
)
_output_img = _result.images[0]

# Save output to Drive for visual inspection
_smoke_out_path = CFG.outputs_eval / "phase2_smoke_output.jpg"
_smoke_out_path.parent.mkdir(parents=True, exist_ok=True)
_output_img.save(_smoke_out_path, "JPEG", quality=95)
print(f"  ok  inference output saved: {_smoke_out_path}")
print(f"      instruction: {_instr[:60]!r}")
print(f"      output size: {_output_img.size}")

# ── 4. Correct CFG demonstration (for Phase 3 reference) ─────────────────
print("\n§19 CFG demonstration (how Phase 3 should call the UNet):")
print("  # Build combined conditioning with CFG enabled:")
print("  cond = build_combined_conditioning([instr], vlm_hs, device, do_cfg=True)")
print("  # → shape (2, 78, 768): rows 0 = uncond, rows 1 = cond")
print("  noise_pred_2b = unet(unet_input_tiled, t, encoder_hidden_states=cond).sample")
print("  # → shape (2, 4, H/8, W/8)")
print("  noise_pred_uncond, noise_pred_cond = noise_pred_2b.chunk(2)")
print("  noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)")
print("  # Proceed with denoising loop using noise_pred")

print("\n✓ §19 PASSED — Phase 2 post-training validation complete.")
print(f"  Checkpoint: {_final_ckpt}")
print(f"  Visual output: {_smoke_out_path}")
print("\n  Phase 2 is ready. Proceed to Phase 3 (inference pipeline).")
print("  Phase 3 will chain: VLM → bbox → SAM2 → SD inpainting")
print("  using VLMProjectionAdapter from this checkpoint.")
